In [1]:
# (1) Install (Kaggle/Colab)
%pip -q install datasetsforecast

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 1.5 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
# (2) Imports + device
import os, math, time, random
from dataclasses import dataclass
from typing import Optional, Tuple

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# --- GPU info / perf knobs (safe)
if torch.cuda.is_available():
    n_gpus = torch.cuda.device_count()
    names = [torch.cuda.get_device_name(i) for i in range(n_gpus)]
    print("gpus:", n_gpus, names)
else:
    print("gpus:", 0)

# Enable faster matmul kernels when available (no-op on some GPUs/torch versions)
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass


device: cpu
gpus: 0


In [3]:
# (3) Seed
def set_seed(seed: int = 0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(0)

In [4]:
# (4) Load LongHorizon2 datasets (ECL / TrafficL / Exchange / Weather)
from datasetsforecast.long_horizon2 import LongHorizon2

def load_longhorizon_group(group: str, data_dir: str = "./data_longhorizon2", normalize: bool = True) -> pd.DataFrame:
    os.makedirs(data_dir, exist_ok=True)
    df = LongHorizon2.load(directory=data_dir, group=group, normalize=normalize)
    # columns: unique_id, ds, y
    return df

In [5]:
# (5) Windowed dataset with (x_cur, x_prev, y_future) for AR
class LongHorizonWindowDataset(Dataset):
    # For each (sid, t):
    #   x_cur  = y[t-input_len : t]                    shape (1, input_len)
    #   x_prev = y[t-input_len-stride_ar : t-stride_ar]
    #   y_fut  = y[t : t+horizon]                      shape (1, horizon)
    #
    # stride_ar is ONLY for the AR term.
    def __init__(
        self,
        df: pd.DataFrame,
        input_len: int,
        horizon: int,
        stride_ar: int,
        split: str = "train",
        train_frac: float = 0.7,
        val_frac: float = 0.1,
        max_per_series: Optional[int] = None,
        seed: int = 0,
    ):
        assert split in ("train", "val", "test")
        self.input_len = int(input_len)
        self.horizon = int(horizon)
        self.stride_ar = int(stride_ar)

        self.series = []
        for uid, g in df.groupby("unique_id"):
            y = g.sort_values("ds")["y"].to_numpy(dtype=np.float32)
            self.series.append(y)

        rng = np.random.RandomState(int(seed))

        self.indices = []
        for sid, y in enumerate(self.series):
            T = len(y)
            t_min = self.input_len + self.stride_ar
            t_max = T - self.horizon
            if t_max <= t_min:
                continue

            n = t_max - t_min
            i_train = t_min + int(train_frac * n)
            i_val   = t_min + int((train_frac + val_frac) * n)

            if split == "train":
                ts = list(range(t_min, i_train))
            elif split == "val":
                ts = list(range(i_train, i_val))
            else:
                ts = list(range(i_val, t_max))

            if max_per_series is not None and len(ts) > int(max_per_series):
                ts = list(rng.choice(ts, size=int(max_per_series), replace=False))

            self.indices.extend([(sid, t) for t in ts])

        if len(self.indices) == 0:
            raise RuntimeError("No samples: check input_len/horizon/stride_ar for this dataset.")

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx: int):
        sid, t = self.indices[idx]
        y = self.series[sid]
        L, S, H = self.input_len, self.stride_ar, self.horizon

        x_cur  = y[t-L : t]
        x_prev = y[t-L-S : t-S]
        y_fut  = y[t : t+H]

        x_cur  = torch.from_numpy(x_cur).unsqueeze(0)   # (1,L)
        x_prev = torch.from_numpy(x_prev).unsqueeze(0)  # (1,L)
        y_fut  = torch.from_numpy(y_fut).unsqueeze(0)   # (1,H)
        return x_cur, x_prev, y_fut

In [6]:
# (6) Bands mu^2 (par neurone) + hinge penalty (abs/rel)
def make_mu2_band(
    n_units: int,
    mu2_target: float,
    lo_k: float = 0.8,
    hi_k: float = 1.6,
    device=device,
    mode: str = "homog",            # "homog" | "hetero"
    hetero_spread: float = 8.0,     # facteur max/min autour de mu2_target (logspace)
    hetero_shuffle: bool = False,
    seed: int = 0,
):
    '''
    Construit une bande [lo_i, hi_i] *par neurone*.

    - mode="homog": lo_i, hi_i constants.
    - mode="hetero": chaque neurone a un budget t_i log-spacé autour de mu2_target:
        t_i = mu2_target * exp(linspace(-log(spread), +log(spread)))
      puis lo_i=lo_k*t_i, hi_i=hi_k*t_i.

    Notes:
    - hétérogène + shuffle=False => hiérarchie stable (souvent mieux empirique).
    - On reste 100% vectorisé (pas de boucle Python par neurone).
    '''
    n_units = int(n_units)
    mu2_target = float(mu2_target)
    lo_k = float(lo_k); hi_k = float(hi_k)

    if mode not in ("homog", "hetero"):
        raise ValueError(f"make_mu2_band: mode must be 'homog' or 'hetero', got {mode}")

    if mode == "homog":
        lo = torch.full((n_units,), lo_k * mu2_target, device=device)
        hi = torch.full((n_units,), hi_k * mu2_target, device=device)
        return lo, hi

    spread = float(max(1.0, hetero_spread))
    # logspace budgets around target
    lo_log = -math.log(spread)
    hi_log = +math.log(spread)
    t = torch.exp(torch.linspace(lo_log, hi_log, steps=n_units, device=device)) * mu2_target

    if hetero_shuffle:
        g = torch.Generator(device=device)
        g.manual_seed(int(seed))
        perm = torch.randperm(n_units, generator=g, device=device)
        t = t[perm]

    lo = lo_k * t
    hi = hi_k * t
    return lo, hi



def hinge_band_penalty(
    mu: torch.Tensor,
    lo,
    hi,
    mode: str = "abs",   # "abs" | "rel" (aliases supported)
    eps: float = 1e-3,   # stabilizer for rel-normalization denominators
    weight_low: float = 1.0,
    weight_high: float = 1.0,
    return_mu2: bool = False,
):
    '''
    Pousse mu2_i = E_b[ mu_{b,i}^2 ] dans [lo_i, hi_i] (par neurone).

    - mode="abs": penalise (lo - mu2)^2 et (mu2 - hi)^2
    - mode="rel": idem mais normalisé par lo/hi, avec eps sécurisé.

    Robustesse (ne casse rien) :
    - accepte lo/hi scalaires, listes, np.ndarray ou torch.Tensor (broadcastables sur (N,))
    - accepte des alias de mode: "absolute"/"relative", espaces, etc.
    - supporte mu de shape (B,N) ou (N,)
    '''
    m = str(mode).lower().strip() if mode is not None else "abs"
    if m in ("absolute", "abs", "l2", "quadratic", "sq"):
        m = "abs"
    elif m in ("relative", "rel", "rel_hinge", "hinge_rel"):
        m = "rel"
    else:
        raise ValueError(f"hinge_band_penalty: mode must be 'abs' or 'rel', got {mode}")

    # mu: (B,N) or (N,)
    if mu.dim() == 1:
        mu2 = mu * mu
    else:
        mu2 = (mu * mu).mean(dim=0)  # (N,)

    # bounds to tensor on the right device/dtype
    if not torch.is_tensor(lo):
        lo = torch.as_tensor(lo, device=mu2.device, dtype=mu2.dtype)
    else:
        lo = lo.to(device=mu2.device, dtype=mu2.dtype)
    if not torch.is_tensor(hi):
        hi = torch.as_tensor(hi, device=mu2.device, dtype=mu2.dtype)
    else:
        hi = hi.to(device=mu2.device, dtype=mu2.dtype)

    # broadcast safety: ensure shapes align (typically (N,))
    # (torch will broadcast automatically when possible)

    low = torch.relu(lo - mu2)
    high = torch.relu(mu2 - hi)

    wl = float(weight_low)
    wh = float(weight_high)

    if m == "abs":
        pen_per = wl * (low * low) + wh * (high * high)
    else:  # m == "rel"
        lo_d = lo.clamp_min(float(eps))
        hi_d = hi.clamp_min(float(eps))
        pen_per = wl * (low / lo_d)**2 + wh * (high / hi_d)**2

    pen = pen_per.mean()
    stats = {
        "mu2_mean": float(mu2.mean().item()),
        "frac_too_low": float((mu2 < lo).float().mean().item()),
        "frac_too_high": float((mu2 > hi).float().mean().item()),
    }

    if return_mu2:
        return pen, stats, mu2.detach()
    return pen, stats


In [7]:
# (7) Projection mu^2 per unit on mu-layer rows (optional, same logic as your MNIST notebook)
@torch.no_grad()
def project_mu2_band_per_unit_on_mu_layer(
    mu_layer: nn.Linear,
    feature_extractor: nn.Module,
    loader: DataLoader,
    lo: torch.Tensor,
    hi: torch.Tensor,
    max_batches: int = 50,
    eps: float = 1e-12,
    max_scale: float = 50.0,
    device=device,
):
    mu_layer.eval()
    feature_extractor.eval()

    mu2_acc = None
    n_seen = 0
    for b, (x_cur, x_prev, y_fut) in enumerate(loader):
        if b >= max_batches:
            break
        x_cur = x_cur.to(device)
        h = feature_extractor(x_cur)      # (B,Hid)
        mu = mu_layer(h)                 # (B,N)
        mu2_b = (mu * mu).mean(dim=0)    # (N,)
        mu2_acc = mu2_b.clone() if mu2_acc is None else (mu2_acc + mu2_b)
        n_seen += 1

    mu2 = (mu2_acc / max(1, n_seen)).clamp_min(eps)
    lo = lo.to(device); hi = hi.to(device)

    s = torch.ones_like(mu2)
    too_low = mu2 < lo
    too_high = mu2 > hi
    s[too_low] = torch.sqrt(lo[too_low] / mu2[too_low])
    s[too_high] = torch.sqrt(hi[too_high] / mu2[too_high])

    if max_scale is not None:
        s = torch.clamp(s, 1.0 / float(max_scale), float(max_scale))

    mu_layer.weight.mul_(s.view(-1, 1))
    mu_layer.bias.mul_(s)

    # quick post-check on same subset
    mu2_post_acc = None
    n_seen2 = 0
    for b, (x_cur, x_prev, y_fut) in enumerate(loader):
        if b >= max_batches:
            break
        x_cur = x_cur.to(device)
        h = feature_extractor(x_cur)
        mu = mu_layer(h)
        mu2_b = (mu * mu).mean(dim=0)
        mu2_post_acc = mu2_b.clone() if mu2_post_acc is None else (mu2_post_acc + mu2_b)
        n_seen2 += 1
    mu2_post = (mu2_post_acc / max(1, n_seen2)).clamp_min(eps)

    return dict(
        mu2_mean_before=float(mu2.mean().item()),
        mu2_mean_after=float(mu2_post.mean().item()),
        frac_too_low=float(too_low.float().mean().item()),
        frac_too_high=float(too_high.float().mean().item()),
        s_min=float(s.min().item()),
        s_max=float(s.max().item()),
        band_mean_width=float((hi - lo).mean().item()),
        band_min_width=float((hi - lo).min().item()),
        band_max_width=float((hi - lo).max().item()),
    )

In [8]:
# (8) Neuron‑VAE layer + models for forecasting
class NeuronVAELayer(nn.Module):
    # q(z|h)=N(mu(h), sigma(h)^2), p(z)=N(0,1)
    # returns z, kl_sum, mu, sigma, kl_per_unit
    def __init__(self, in_dim: int, n_units: int, sigma_mode: str = "scalar", sigma_init: float = 1.0):
        super().__init__()
        self.in_dim = int(in_dim)
        self.n_units = int(n_units)
        self.mu = nn.Linear(self.in_dim, self.n_units)

        assert sigma_mode in ("scalar", "per_unit", "per_unit_input")
        self.sigma_mode = sigma_mode

        if sigma_mode == "scalar":
            self.log_sigma = nn.Parameter(torch.tensor(math.log(float(sigma_init)), dtype=torch.float32))
        elif sigma_mode == "per_unit":
            self.log_sigma = nn.Parameter(torch.full((self.n_units,), math.log(float(sigma_init)), dtype=torch.float32))
        else:
            self.log_sigma_layer = nn.Linear(self.in_dim, self.n_units)
            nn.init.constant_(self.log_sigma_layer.bias, math.log(float(sigma_init)))

    def forward(self, h: torch.Tensor):
        mu = self.mu(h)  # (B,N)

        if self.sigma_mode == "scalar":
            sigma = torch.exp(self.log_sigma).clamp(1e-4, 10.0)
            sigma2 = sigma * sigma
            kl_per_unit = 0.5 * (mu * mu + sigma2 - torch.log(sigma2) - 1.0)
        elif self.sigma_mode == "per_unit":
            sigma = torch.exp(self.log_sigma).clamp(1e-4, 10.0).view(1, -1)
            sigma2 = sigma * sigma
            kl_per_unit = 0.5 * (mu * mu + sigma2 - torch.log(sigma2) - 1.0)
        else:
            log_sigma = self.log_sigma_layer(h).clamp(math.log(1e-4), math.log(10.0))
            sigma = torch.exp(log_sigma)
            sigma2 = sigma * sigma
            kl_per_unit = 0.5 * (mu * mu + sigma2 - torch.log(sigma2) - 1.0)

        eps = torch.randn_like(mu)
        z = mu + torch.sqrt(torch.clamp(sigma2, min=1e-8)) * eps

        kl_sum = kl_per_unit.sum(dim=1)
        return z, kl_sum, mu, sigma, kl_per_unit


class FeatureMLP(nn.Module):
    # x_cur: (B,1,L) -> h: (B,hidden)
    def __init__(self, input_len: int, hidden: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(int(input_len), int(hidden)),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor):
        x = x.squeeze(1)  # (B,L)
        return self.net(x)


class ModelDeterministicTS(nn.Module):
    def __init__(self, input_len: int, hidden: int, n_units: int, horizon: int):
        super().__init__()
        self.feat = FeatureMLP(input_len, hidden)
        self.act = nn.Linear(hidden, n_units)
        self.head = nn.Linear(n_units, horizon)
        self.n_units = int(n_units)

    def forward(self, x_cur: torch.Tensor):
        h = self.feat(x_cur)
        a = self.act(h)
        yhat = self.head(a / math.sqrt(self.n_units))
        aux = {
            "kl": torch.zeros(x_cur.size(0), device=x_cur.device),
            "kl_per_unit": torch.zeros(x_cur.size(0), self.n_units, device=x_cur.device),
            "mu": a,
            "sigma": torch.zeros((), device=x_cur.device),
            "z": a,  # unused
        }
        return yhat, aux


class ModelNeuronVAETS(nn.Module):
    def __init__(self, input_len: int, hidden: int, n_units: int, horizon: int, sigma_mode: str, sigma_init: float):
        super().__init__()
        self.feat = FeatureMLP(input_len, hidden)
        self.neuron = NeuronVAELayer(hidden, n_units, sigma_mode=sigma_mode, sigma_init=sigma_init)
        self.head = nn.Linear(n_units, horizon)
        self.n_units = int(n_units)

    def forward(self, x_cur: torch.Tensor):
        h = self.feat(x_cur)
        z, kl_sum, mu, sigma, kl_per_unit = self.neuron(h)
        yhat = self.head(z / math.sqrt(self.n_units))
        aux = {"kl": kl_sum, "kl_per_unit": kl_per_unit, "mu": mu, "sigma": sigma, "z": z}
        return yhat, aux

In [9]:
# (9) Per‑unit beta thermostat (fixes your NameError: PerUnitBetaController undefined)
class PerUnitBetaController:
    def __init__(
        self,
        n_units: int,
        beta_init: float = 1e-3,
        beta_min: float = 1e-6,
        beta_max: float = 1e-1,
        beta_ratio_min: float = 0.25,
        beta_ratio_max: float = 4.0,
        C_min: float = 0.05,
        C_max: float = 0.30,
        kl_eps: float = 1e-4,
        ema_alpha: float = 0.1,
        lr_logbeta: float = 0.2,
        alive_min: float = 0.05,
        emergency_factor: float = 0.5,
        device: torch.device = device,
    ):
        self.n_units = int(n_units)
        self.beta_min = float(beta_min)
        self.beta_max = float(beta_max)
        self.beta_ratio_min = float(beta_ratio_min)
        self.beta_ratio_max = float(beta_ratio_max)
        self.C_min = float(C_min)
        self.C_max = float(C_max)
        self.kl_eps = float(kl_eps)
        self.ema_alpha = float(ema_alpha)
        self.lr_logbeta = float(lr_logbeta)
        self.alive_min = float(alive_min)
        self.emergency_factor = float(emergency_factor)
        self.device = device

        init = min(max(float(beta_init), self.beta_min), self.beta_max)
        self.log_beta = torch.full((self.n_units,), math.log(init), device=self.device)
        self.kl_ema = None
        self.alive_ema = None

    def beta_vec(self) -> torch.Tensor:
        return torch.exp(self.log_beta).clamp(self.beta_min, self.beta_max)

    @torch.no_grad()
    def update(self, kl_bj: torch.Tensor):
        # kl_bj: (B,N)
        kl_bj = kl_bj.detach()
        alive_j = (kl_bj > self.kl_eps).float().mean(dim=0)
        kl_eff_j = kl_bj.mean(dim=0)

        if self.kl_ema is None:
            self.kl_ema = kl_eff_j.clone()
            self.alive_ema = alive_j.clone()
        else:
            a = self.ema_alpha
            self.kl_ema = (1 - a) * self.kl_ema + a * kl_eff_j
            self.alive_ema = (1 - a) * self.alive_ema + a * alive_j

        err = torch.zeros_like(self.kl_ema)
        err = torch.where(self.kl_ema < self.C_min, self.kl_ema - self.C_min, err)
        err = torch.where(self.kl_ema > self.C_max, self.kl_ema - self.C_max, err)

        emergency = (self.alive_ema < self.alive_min).float()
        err = err - emergency * (err.abs() + self.emergency_factor)

        self.log_beta = self.log_beta + self.lr_logbeta * err
        b = self.beta_vec()

        # --- prevent per-unit beta explosions / collapses (keeps all units "alive")
        b_mean = float(b.mean().item())
        if math.isfinite(b_mean) and b_mean > 0:
            lo = max(self.beta_min, b_mean * self.beta_ratio_min)
            hi = min(self.beta_max, b_mean * self.beta_ratio_max)
            b = b.clamp(lo, hi)

        self.log_beta = torch.log(b)

        return b, {
            "kl_mean": float(self.kl_ema.mean().item()),
            "alive_mean": float(self.alive_ema.mean().item()),
            "beta_mean": float(b.mean().item()),
            "beta_min": float(b.min().item()),
            "beta_max": float(b.max().item()),
        }

In [10]:
# (9b) Per-neuron VAE homeostasis (vectorisé) — "neurone VAE vivant"
@torch.no_grad()
def homeostatic_rescale_mu_layer_from_mu2(
    mu_layer: nn.Linear,
    mu2: torch.Tensor,     # (N,) estimate (batch/EMA)
    lo: torch.Tensor,      # (N,)
    hi: torch.Tensor,      # (N,)
    eta: float = 0.10,     # vitesse (0 = off, 1 = projection dure)
    max_scale_step: float = 1.5,
    inside_target: str = "mu2",  # "mu2" | "center_geom" | "center_arith"
    eps: float = 1e-12,
):
    '''
    Homeostasie douce: rapproche mu2_i vers la bande [lo_i,hi_i] en rescalant
    *les lignes* de la couche mu (poids + biais), mais par petites étapes.

    scale_i = (target_i / mu2_i)^(0.5*eta).

    inside_target:
      - "mu2"          : no-op si mu2 est dans la bande (comportement legacy)
      - "center_geom"  : nudges vers sqrt(lo*hi) même si mu2 est dans la bande
      - "center_arith" : nudges vers 0.5*(lo+hi) même si mu2 est dans la bande
    '''
    mu2 = mu2.detach().clamp_min(eps)
    lo = lo.to(mu2.device)
    hi = hi.to(mu2.device)

    s = torch.ones_like(mu2)
    too_low = mu2 < lo
    too_high = mu2 > hi
    # choose inside-band target
    if inside_target == "mu2":
        inside = mu2
    elif inside_target == "center_geom":
        inside = torch.sqrt(lo.clamp_min(eps) * hi.clamp_min(eps))
    elif inside_target == "center_arith":
        inside = 0.5 * (lo + hi)
    else:
        raise ValueError(f"homeostat: inside_target must be 'mu2'|'center_geom'|'center_arith', got {inside_target}")

    target = torch.where(too_low, lo, torch.where(too_high, hi, inside))
    # eta=1 => projection; eta<1 => pas
    s = torch.pow(target / mu2, 0.5 * float(eta))

    if max_scale_step is not None:
        m = float(max_scale_step)
        s = torch.clamp(s, 1.0 / m, m)

    mu_layer.weight.mul_(s.view(-1, 1))
    mu_layer.bias.mul_(s)

    return {
        "frac_too_low": float(too_low.float().mean().item()),
        "frac_too_high": float(too_high.float().mean().item()),
        "s_min": float(s.min().item()),
        "s_max": float(s.max().item()),
        "mu2_mean_before": float(mu2.mean().item()),
        "mu2_mean_target": float(target.mean().item()),
    }


class PerNeuronVAERegulator:
    '''
    Régulateur *par neurone* (vectorisé) pour un NeuronVAELayer:
      - Maintient mu2_i dans une bande [lo_i, hi_i] via homeostasie (rescale mu-layer rows)
      - Maintient KL_i dans [C_min, C_max] via beta thermostat (déjà présent)
      - Empêche sigma d'aller dans des zones absurdes (clamp + "kick" anti-collapse)
      - Option: ajuste lam_ar pour que l'AR ne vampirise pas la loss
    '''
    def __init__(
        self,
        n_units: int,
        lo: torch.Tensor,
        hi: torch.Tensor,
        mu_layer: nn.Linear,
        neuron_layer: NeuronVAELayer,
        beta_ctl: Optional[PerUnitBetaController],
        device=device,
        # --- mu2 homeostat
        use_mu_homeostat: bool = True,
        eta_mu: float = 0.10,
        max_scale_step: float = 1.5,
        mu_ema_alpha: float = 0.05,
        # --- sigma guard
        use_sigma_guard: bool = True,
        sigma_floor: float = 0.15,
        sigma_ceil: float = 5.0,
        sigma_kick: float = 0.02,   # log-space additive kick
        alive_mu2_eps: float = 1e-4,
        alive_kl_eps: float = 1e-4,
        alive_min: float = 0.05,
        # --- AR share controller (optional)
        adapt_lam_ar: bool = False,
        ar_share_target: float = 0.15,
        ar_lr: float = 0.10,
        lam_ar_min: float = 1e-4,
        lam_ar_max: float = 10.0,
        inside_target: str = "mu2",  # "mu2" | "center_geom" | "center_arith"
    ):
        self.n_units = int(n_units)
        self.device = device
        self.lo = lo.to(device)
        self.hi = hi.to(device)
        self.mu_layer = mu_layer
        self.neuron_layer = neuron_layer
        self.beta_ctl = beta_ctl

        self.use_mu_homeostat = bool(use_mu_homeostat)
        self.eta_mu = float(eta_mu)
        self.max_scale_step = float(max_scale_step)
        self.inside_target = str(inside_target)
        self.mu_ema_alpha = float(mu_ema_alpha)

        self.use_sigma_guard = bool(use_sigma_guard)
        self.sigma_floor = float(sigma_floor)
        self.sigma_ceil = float(sigma_ceil)
        self.sigma_kick = float(sigma_kick)

        self.alive_mu2_eps = float(alive_mu2_eps)
        self.alive_kl_eps = float(alive_kl_eps)
        self.alive_min = float(alive_min)

        self.adapt_lam_ar = bool(adapt_lam_ar)
        self.ar_share_target = float(ar_share_target)
        self.ar_lr = float(ar_lr)
        self.lam_ar_min = float(lam_ar_min)
        self.lam_ar_max = float(lam_ar_max)

        # EMAs / buffers
        self.mu2_ema = None
        self.kl_ema = None
        self.alive_ema = None
        self.last_kl_bj = None

        self.mse_ema = None
        self.klw_ema = None
        self.bandw_ema = None
        self.ar_ema = None

        self.last_act = None

    @torch.no_grad()
    def observe(self, mu: torch.Tensor, kl_per_unit: torch.Tensor, sigma: Optional[torch.Tensor] = None,
                mse: Optional[torch.Tensor] = None, ar: Optional[torch.Tensor] = None,
                kl_w: Optional[torch.Tensor] = None, band_w: Optional[torch.Tensor] = None):
        '''
        Appelé à chaque batch (ou fréquemment). Tout est vectorisé.
        mu: (B,N)
        kl_per_unit: (B,N)
        sigma: (1,N) ou (B,N) ou None
        '''
        mu2 = (mu.detach() * mu.detach()).mean(dim=0)  # (N,)
        kl_j = kl_per_unit.detach().mean(dim=0)        # (N,)

        alive_j = ((mu2 > self.alive_mu2_eps) & (kl_j > self.alive_kl_eps)).float()

        a = self.mu_ema_alpha
        if self.mu2_ema is None:
            self.mu2_ema = mu2.clone()
            self.kl_ema = kl_j.clone()
            self.alive_ema = alive_j.clone()
        else:
            self.mu2_ema = (1 - a) * self.mu2_ema + a * mu2
            self.kl_ema = (1 - a) * self.kl_ema + a * kl_j
            self.alive_ema = (1 - a) * self.alive_ema + a * alive_j

        self.last_kl_bj = kl_per_unit.detach()

        # loss component EMAs (for optional AR share control)
        if kl_w is not None:
            k = float(kl_w.detach().item())
            self.klw_ema = k if self.klw_ema is None else (0.9 * self.klw_ema + 0.1 * k)
        if band_w is not None:
            bw = float(band_w.detach().item())
            self.bandw_ema = bw if self.bandw_ema is None else (0.9 * self.bandw_ema + 0.1 * bw)
        if mse is not None:
            m = float(mse.detach().item())
            self.mse_ema = m if self.mse_ema is None else (0.9 * self.mse_ema + 0.1 * m)
        if ar is not None:
            a2 = float(ar.detach().item())
            self.ar_ema = a2 if self.ar_ema is None else (0.9 * self.ar_ema + 0.1 * a2)

    @torch.no_grad()
    def act(self, cfg: Optional['Cfg'] = None, warmup_done: bool = True, ar_mult: float = 1.0):
        '''
        Applique les corrections (homeostasie mu, update beta, sigma-guard, adapt lam_ar).
        Appelé périodiquement (par ex. toutes les X itérations ou à chaque epoch).
        '''
        info = {}

        # --- beta thermostat: KL_i in [C_min,C_max] + emergency alive
        if (self.beta_ctl is not None) and (self.last_kl_bj is not None) and warmup_done:
            beta_vec, beta_info = self.beta_ctl.update(self.last_kl_bj)
            info["beta"] = beta_info
        else:
            beta_vec = None

        # --- mu2 homeostat: keep mu2_i in band (adaptive strength)
        if self.use_mu_homeostat and (self.mu2_ema is not None) and warmup_done:
            # If many units are out-of-band, increase the correction strength a bit (stable cap).
            frac_out = float((((self.mu2_ema < self.lo) | (self.mu2_ema > self.hi)).float().mean().item()))
            eta_eff = float(self.eta_mu) * (1.0 + 2.0 * frac_out)
            eta_eff = min(0.50, max(0.0, eta_eff))

            info["mu_homeostat"] = homeostatic_rescale_mu_layer_from_mu2(
                self.mu_layer, self.mu2_ema, self.lo, self.hi,
                eta=eta_eff, max_scale_step=self.max_scale_step,
                inside_target=self.inside_target
            )
            info["mu_homeostat"]["eta_eff"] = float(eta_eff)

        # --- sigma guard: clamp + tiny kick if many units dead
        if self.use_sigma_guard and warmup_done:
            try:
                if hasattr(self.neuron_layer, "log_sigma") and isinstance(self.neuron_layer.log_sigma, torch.nn.Parameter):
                    lo_ls = math.log(self.sigma_floor)
                    hi_ls = math.log(self.sigma_ceil)
                    self.neuron_layer.log_sigma.data.clamp_(lo_ls, hi_ls)

                    # anti-collapse kick (per-unit only)
                    if self.neuron_layer.sigma_mode == "per_unit" and (self.alive_ema is not None):
                        dead = (self.alive_ema < self.alive_min).float()
                        if float(dead.mean().item()) > 0:
                            self.neuron_layer.log_sigma.data.add_(dead * float(self.sigma_kick))
                            self.neuron_layer.log_sigma.data.clamp_(lo_ls, hi_ls)
                            info["sigma_kick_frac"] = float(dead.mean().item())
            except Exception as e:
                info["sigma_guard_error"] = str(e)

        # --- adapt lam_ar: keep AR share bounded (FIXED SIGN + safety cap)
        if self.adapt_lam_ar and warmup_done and (cfg is not None) and (self.mse_ema is not None) and (self.ar_ema is not None):
            klw = 0.0 if self.klw_ema is None else float(self.klw_ema)
            bw = 0.0 if self.bandw_ema is None else float(self.bandw_ema)

            lam = float(cfg.lam_ar)
            ar_mult = float(ar_mult)
            lam_eff = lam * ar_mult
            denom = max(1e-12, float(self.mse_ema) + klw + bw + lam_eff * float(self.ar_ema))
            share = (lam_eff * float(self.ar_ema)) / denom  # in [0,1]

            target = float(self.ar_share_target)
            cap = getattr(cfg, "ar_share_cap", None)
            cap = None if cap is None else float(cap)

            # If share > target => decrease lam_ar; if share < target => increase lam_ar.
            err = share - target
            new_lam = lam * math.exp(-float(self.ar_lr) * err)

            # Extra safety: if cap exists and share exceeds it, damp more aggressively.
            if (cap is not None) and (share > cap):
                new_lam = lam * math.exp(-2.0 * float(self.ar_lr) * (share - cap))

            new_lam = min(max(new_lam, self.lam_ar_min), self.lam_ar_max)

            info["ar_share"] = float(share)
            info["ar_share_target"] = float(target)
            info["ar_mult"] = float(ar_mult)
            if cap is not None:
                info["ar_share_cap"] = float(cap)
            info["lam_ar_old"] = float(lam)
            info["lam_ar_new"] = float(new_lam)

            cfg.lam_ar = float(new_lam)

        self.last_act = info
        return beta_vec, info


In [11]:
# (10) AR penalty (extracted from your NumPy toy, PyTorch version)
def ar_sigma_from_band(band_width_mean: float, ar_k: float, floor: float = 0.03, ceil: float = 1.0) -> float:
    s = float(ar_k) * float(band_width_mean)
    s = min(max(s, float(floor)), float(ceil))
    return float(s)

def ar_penalty(cur: torch.Tensor, prev: torch.Tensor, phi: float, ar_sigma: float):
    resid = cur - float(phi) * prev
    return (resid * resid).mean() / (2.0 * (float(ar_sigma) ** 2) + 1e-12)
# --- Better AR sigma: use mu-scale derived from the current mu^2 band (stable across hetero bands)
def ar_sigma_from_band_mu2(
    lo: torch.Tensor,
    hi: torch.Tensor,
    ar_k: float,
    floor: float = 0.03,
    ceil: float = 1.0,
    eps: float = 1e-12,
    center: str = "geom",  # "geom" (sqrt(lo*hi)) or "arith" (0.5*(lo+hi))
) -> float:
    """Return a single (global) AR sigma on the *mu/z scale*.

    We derive a representative mu^2 level from the band (per-unit [lo_i,hi_i]), then convert to mu-scale:
      mu2_center_i = sqrt(lo_i*hi_i)  (geom)  or  0.5*(lo_i+hi_i) (arith)
      mu_scale ≈ sqrt(mean(mu2_center_i))
      ar_sigma = ar_k * mu_scale, clamped.

    This avoids the unit-mismatch of using a *mu^2 width* as a *mu scale*.
    """
    if not torch.is_tensor(lo):
        lo = torch.as_tensor(lo)
    if not torch.is_tensor(hi):
        hi = torch.as_tensor(hi)
    lo = lo.detach()
    hi = hi.detach()
    if str(center).lower().startswith("arith"):
        mu2_center = 0.5 * (lo + hi)
    else:
        mu2_center = torch.sqrt(lo.clamp_min(eps) * hi.clamp_min(eps))
    mu2_ref = float(mu2_center.mean().clamp_min(eps).item())
    mu_ref = math.sqrt(mu2_ref)
    s = float(ar_k) * float(mu_ref)
    s = min(max(s, float(floor)), float(ceil))
    return float(s)

In [12]:
# (11) Config
@dataclass
class Cfg:
    group: str = "ECL"         # "TrafficL", "Exchange", "Weather"
    select_by: str = "mse"  # "mse" | "probml" (val loss)
    input_len: int = 336
    horizon: int = 96
    stride_ar: int = 4

    hidden: int = 256
    n_units: int = 1024
    model: str = "nvae"        # "det" or "nvae"
    sigma_mode: str = "per_unit"
    sigma_init: float = 0.5

    # --- neurone-VAE vivant: bande mu^2 par neurone
    mu2_target: float = 0.03
    band_mode: str = "hetero"          # "homog" | "hetero"
    band_lo_k: float = 0.80
    band_hi_k: float = 1.60
    hetero_spread: float = 8.0
    hetero_shuffle: bool = False
    band_penalty_mode: str = "rel"     # "abs" | "rel"
    lam_band: float = 3.0             # en mode 'rel' on peut baisser fortement

    band_rel_eps: float = 1e-3          # eps pour mode 'rel' (évite explosion sur petits budgets)
    band_ramp_steps: int = 200         # ramp de la pénalité bande après warmup (0 => step)


    # --- band auto-calibration (garde l'hétérogénéité, ajuste juste l'échelle globale)
    use_band_calib: bool = True
    band_calib_every: int = 200        # en steps (0 => off)
    band_calib_ema: float = 0.70       # EMA en log-space
    band_target_low: float = 0.20      # fraction voulue sous lo (approx)
    band_target_high: float = 0.10     # fraction voulue au-dessus de hi (approx)
    band_scale_min: float = 0.02
    band_scale_max: float = 5.00
    band_calib_step_cap: float = 1.25   # max multiplicative change per calib step
    
    # --- focus bas (anti latent-OFF) : au début on pénalise moins la partie haute
    band_focus_low: bool = True
    band_low_focus_thresh: float = 0.35
    lam_band_hi_mult: float = 0.30

    # --- legacy projection (garde-fou). Le mode recommandé est homeostat (plus doux).
    use_projection: bool = False
    proj_every: int = 1
    proj_batches: int = 50
    proj_max_scale: float = 50.0

    # --- beta thermostat (par neurone)
    use_beta_thermostat: bool = True
    beta_init: float = 1e-3
    beta_min: float = 1e-6
    beta_max: float = 1e-2
    beta_ratio_min: float = 0.25   # clamp per-unit beta around mean
    beta_ratio_max: float = 4.0
    C_min: float = 0.05
    C_max: float = 0.30
    lr_logbeta: float = 0.2

    # --- homeostasie neurone: autosauvegarde anti-collapse (par neurone)
    use_neuron_regulator: bool = True
    warmup_steps: int = 200          # démarrage: on laisse apprendre avant régulation
    ctrl_every: int = 10             # fréquence des actions (en steps)
    eta_mu: float = 0.15             # vitesse homeostat mu^2 (0.05-0.3 raisonnable)
    max_scale_step: float = 2.0      # clamp du rescale par action
    homeo_inside_target: str = "center_geom"  # "mu2"|"center_geom"|"center_arith"
    sigma_floor: float = 0.15
    sigma_ceil: float = 5.0
    sigma_kick: float = 0.02         # kick log-sigma si neurones morts
    alive_min: float = 0.05          # seuil "neurone vivant"

    # --- AR latent
    use_ar: bool = False
    ar_on_mu: bool = True
    phi: float = 0.95
    ar_k: float = 0.8
    lam_ar: float = 1.0

    ar_sigma_floor: float = 0.03     # prevents AR penalty blow-up when bands shrink
    ar_sigma_ceil: float = 1.0

    ar_ramp_steps: int = 200           # ramp AR après warmup (0 => step)

    # auto: empêcher AR de vampiriser la loss
    adapt_lam_ar: bool = True
    ar_share_target: float = 0.20
    ar_share_cap: float = 0.30   # cap dur de sécurité (0.25–0.35 typ.)
    ar_lr: float = 0.10
    lam_ar_min: float = 0.25
    lam_ar_max: float = 10.0
    # --- perf (Kaggle GPU)
    use_amp: bool = True                 # mixed precision (fp16) when CUDA
    use_data_parallel: bool = True       # use 2xGPU via nn.DataParallel when available
    num_workers: int = 2                 # DataLoader workers (try 2 or 4 on Kaggle)
    prefetch_factor: int = 2
    persistent_workers: bool = True


    # --- training
    epochs: int = 10
    early_stop_patience: int = 6
    early_stop_min_delta: float = 1e-4
    batch_size: int = 256
    lr: float = 1e-3
    weight_decay: float = 0.0
    clip_grad: float = 1.0

cfg = Cfg()
print(cfg)


Cfg(group='ECL', select_by='mse', input_len=336, horizon=96, stride_ar=4, hidden=256, n_units=1024, model='nvae', sigma_mode='per_unit', sigma_init=0.5, mu2_target=0.03, band_mode='hetero', band_lo_k=0.8, band_hi_k=1.6, hetero_spread=8.0, hetero_shuffle=False, band_penalty_mode='rel', lam_band=3.0, band_rel_eps=0.001, band_ramp_steps=200, use_band_calib=True, band_calib_every=200, band_calib_ema=0.7, band_target_low=0.2, band_target_high=0.1, band_scale_min=0.02, band_scale_max=5.0, band_calib_step_cap=1.25, band_focus_low=True, band_low_focus_thresh=0.35, lam_band_hi_mult=0.3, use_projection=False, proj_every=1, proj_batches=50, proj_max_scale=50.0, use_beta_thermostat=True, beta_init=0.001, beta_min=1e-06, beta_max=0.01, beta_ratio_min=0.25, beta_ratio_max=4.0, C_min=0.05, C_max=0.3, lr_logbeta=0.2, use_neuron_regulator=True, warmup_steps=200, ctrl_every=10, eta_mu=0.15, max_scale_step=2.0, homeo_inside_target='center_geom', sigma_floor=0.15, sigma_ceil=5.0, sigma_kick=0.02, alive_mi

In [13]:
# (12) DataLoaders (subset sizes to keep runs quick) — GPU-friendly
import torch, random
from torch.utils.data import DataLoader

df = load_longhorizon_group(cfg.group)

train_ds = LongHorizonWindowDataset(df, cfg.input_len, cfg.horizon, cfg.stride_ar, split="train", max_per_series=400, seed=0)
val_ds   = LongHorizonWindowDataset(df, cfg.input_len, cfg.horizon, cfg.stride_ar, split="val",   max_per_series=200, seed=1)
test_ds  = LongHorizonWindowDataset(df, cfg.input_len, cfg.horizon, cfg.stride_ar, split="test",  max_per_series=200, seed=2)

def _seed_worker(worker_id: int):
    # make DataLoader workers deterministic-ish
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(0)

bs = int(getattr(cfg, "batch_size", 256))
nw = int(getattr(cfg, "num_workers", 0))
if not torch.cuda.is_available():
    nw = 0

common = dict(
    batch_size=bs,
    num_workers=nw,
    pin_memory=torch.cuda.is_available(),
)
if nw > 0:
    common.update(
        prefetch_factor=int(getattr(cfg, "prefetch_factor", 2)),
        persistent_workers=bool(getattr(cfg, "persistent_workers", True)),
        worker_init_fn=_seed_worker,
        generator=g,
    )

train_loader = DataLoader(train_ds, shuffle=True,  drop_last=True,  **common)
val_loader   = DataLoader(val_ds,   shuffle=False, drop_last=False, **common)
test_loader  = DataLoader(test_ds,  shuffle=False, drop_last=False, **common)

print("sizes:", len(train_ds), len(val_ds), len(test_ds), "| batch_size:", bs, "| num_workers:", nw)


100%|██████████| 54.0M/54.0M [00:00<00:00, 68.1MiB/s]
INFO:datasetsforecast.utils:Successfully downloaded all_six_datasets.zip, 53999897, bytes.
INFO:datasetsforecast.utils:Decompressing zip file...
INFO:datasetsforecast.utils:Successfully decompressed data_longhorizon2/longhorizon2/all_six_datasets.zip


sizes: 128400 64200 64200 | batch_size: 256 | num_workers: 0


In [14]:
# (13) Model + bands + beta controller + per-neuron regulator
if cfg.model == "det":
    model = ModelDeterministicTS(cfg.input_len, cfg.hidden, cfg.n_units, cfg.horizon).to(device)
else:
    model = ModelNeuronVAETS(cfg.input_len, cfg.hidden, cfg.n_units, cfg.horizon, cfg.sigma_mode, cfg.sigma_init).to(device)

opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)


# --- per-neuron band (base) + échelle globale (auto-calibration)
base_lo, base_hi = make_mu2_band(
    cfg.n_units, cfg.mu2_target,
    lo_k=cfg.band_lo_k, hi_k=cfg.band_hi_k,
    mode=cfg.band_mode,
    hetero_spread=cfg.hetero_spread,
    hetero_shuffle=cfg.hetero_shuffle,
    seed=0,
    device=device
)

band_state = {"s_lo": 1.0, "s_hi": 1.0}  # scalaires (ajustés automatiquement)

def current_band():
    lo_eff = base_lo * float(band_state["s_lo"])
    hi_eff = base_hi * float(band_state["s_hi"])
    # sécurité: garantit hi > lo
    hi_eff = torch.maximum(hi_eff, lo_eff * 1.05)
    return lo_eff, hi_eff

lo, hi = current_band()
band_width_mean = float((hi - lo).mean().item())
ar_sigma = ar_sigma_from_band_mu2(lo, hi, cfg.ar_k, cfg.ar_sigma_floor, cfg.ar_sigma_ceil)

# --- per-unit beta thermostat
beta_ctl = None
beta_vec = None
if (cfg.model != "det") and cfg.use_beta_thermostat:
    beta_ctl = PerUnitBetaController(
        n_units=cfg.n_units,
        beta_init=cfg.beta_init, beta_min=cfg.beta_min, beta_max=cfg.beta_max,
        beta_ratio_min=cfg.beta_ratio_min, beta_ratio_max=cfg.beta_ratio_max,
        C_min=cfg.C_min, C_max=cfg.C_max,
        lr_logbeta=cfg.lr_logbeta,
        alive_min=cfg.alive_min,
        device=device
    )
    beta_vec = beta_ctl.beta_vec()

# --- per-neuron regulator (homeostasie)
reg = None
if (cfg.model != "det") and cfg.use_neuron_regulator:
    reg = PerNeuronVAERegulator(
        n_units=cfg.n_units,
        lo=lo, hi=hi,
        mu_layer=model.neuron.mu,
        neuron_layer=model.neuron,
        beta_ctl=beta_ctl,
        device=device,
        eta_mu=cfg.eta_mu,
        max_scale_step=cfg.max_scale_step,
        inside_target=cfg.homeo_inside_target,
        sigma_floor=cfg.sigma_floor,
        sigma_ceil=cfg.sigma_ceil,
        sigma_kick=cfg.sigma_kick,
        alive_min=cfg.alive_min,
        adapt_lam_ar=cfg.adapt_lam_ar,
        ar_share_target=cfg.ar_share_target,
        ar_lr=cfg.ar_lr,
        lam_ar_min=cfg.lam_ar_min,
        lam_ar_max=cfg.lam_ar_max,
    )

print("band_width_mean:", band_width_mean, "ar_sigma:", ar_sigma)
print("beta_vec:", None if beta_vec is None else (float(beta_vec.mean()), float(beta_vec.min()), float(beta_vec.max())))
print("regulator:", "ON" if reg is not None else "OFF")


band_width_mean: 0.045495789498090744 ar_sigma: 0.20292384824897094
beta_vec: (0.0010000000474974513, 0.0009999999310821295, 0.0009999999310821295)
regulator: ON


In [15]:
# (14) Eval (corrected) — uses per-unit beta_vec and AR
@torch.no_grad()
def eval_epoch(loader, *, band_mult: float = 1.0, ar_mult: float = 1.0):
    model.eval()
    total = 0
    mse_sum, mae_sum = 0.0, 0.0
    loss_sum, kl_sum, band_sum, ar_sum = 0.0, 0.0, 0.0, 0.0
    mu2_mean_acc, low_acc, high_acc = 0.0, 0.0, 0.0
    n_batches = 0

    band_mult = float(band_mult)
    ar_mult = float(ar_mult)
    lam_band_eff = float(cfg.lam_band) * band_mult
    lam_ar_eff   = float(cfg.lam_ar)   * ar_mult
    for x_cur, x_prev, y_fut in loader:
        x_cur, x_prev, y_fut = x_cur.to(device), x_prev.to(device), y_fut.to(device)
        y_true = y_fut.squeeze(1)  # (B,H)

        y_hat, aux = model(x_cur)
        mse = F.mse_loss(y_hat, y_true)
        mae = F.l1_loss(y_hat, y_true)
        loss = mse

        kl = aux["kl"].mean()
        if (cfg.model != "det") and (beta_vec is not None):
            kl_bj = aux["kl_per_unit"]
            reg = (kl_bj * beta_vec.view(1, -1)).sum(dim=1).mean()
            loss = loss + reg

        band = torch.tensor(0.0, device=device)
        stats = {"mu2_mean": 0.0, "frac_too_low": 0.0, "frac_too_high": 0.0}
        if cfg.model != "det":
            lo_eval, hi_eval = current_band()
            band, stats = hinge_band_penalty(aux["mu"], lo_eval, hi_eval, mode=cfg.band_penalty_mode, eps=cfg.band_rel_eps)
            loss = loss + lam_band_eff * band

        ar = torch.tensor(0.0, device=device)
        if (cfg.model != "det") and cfg.use_ar:
            _, aux_prev = model(x_prev)
            cur = aux["mu"] if cfg.ar_on_mu else aux["z"]
            prv = aux_prev["mu"] if cfg.ar_on_mu else aux_prev["z"]
            ar = ar_penalty(cur, prv, phi=cfg.phi, ar_sigma=ar_sigma)
            loss = loss + lam_ar_eff * ar

        B = x_cur.size(0)
        total += B
        mse_sum += float(mse.item()) * B
        mae_sum += float(mae.item()) * B
        loss_sum += float(loss.item()) * B
        kl_sum += float(kl.item()) * B
        band_sum += float(band.item()) * B
        ar_sum += float(ar.item()) * B

        mu2_mean_acc += stats["mu2_mean"]
        low_acc += stats["frac_too_low"]
        high_acc += stats["frac_too_high"]
        n_batches += 1

    return {
        "loss": loss_sum / max(1, total),
        "mse": mse_sum / max(1, total),
        "mae": mae_sum / max(1, total),
        "kl": kl_sum / max(1, total),
        "band": band_sum / max(1, total),
        "ar": ar_sum / max(1, total),
        "mu2_mean": mu2_mean_acc / max(1, n_batches),
        "frac_too_low": low_acc / max(1, n_batches),
        "frac_too_high": high_acc / max(1, n_batches),
        "band_mult": band_mult,
        "ar_mult": ar_mult,
        "lam_band_eff": lam_band_eff,
        "lam_ar_eff": lam_ar_eff,
    }

In [16]:
# (15) Train loop — neurone-vivant (homeostat) + beta thermostat + logs

def probml_proxy(summary: dict, cfg: Cfg) -> float:
    """Paper-friendly proxy: enforce constraints first, then MSE."""
    low_t = float(cfg.band_target_low)
    high_t = float(cfg.band_target_high)
    # AR share computed from current lam_ar and summary parts (rough, but consistent)
    denom = max(1e-12, float(summary.get("loss", 0.0)))
    ar_share = 0.0
    if cfg.use_ar:
        ar_share = (float(summary.get('lam_ar_eff', cfg.lam_ar)) * float(summary.get('ar', 0.0))) / denom  # in [0,1]-ish

    p = 0.0
    p += max(0.0, float(summary.get("frac_too_low", 1.0)) - low_t) ** 2
    p += max(0.0, float(summary.get("frac_too_high", 1.0)) - high_t) ** 2
    p += max(0.0, ar_share - float(cfg.ar_share_cap)) ** 2

    # Weighted sum (constraints dominate)
    return float(summary.get("mse", 1e18)) + 10.0 * p

history = []
best_val = 1e18
best_state = None
best_bundle = None  # stores model + autopilot state (band_state, ar_sigma, beta_vec, multipliers)
best_band_state = None
best_beta_vec = None
best_ar_sigma = None
best_band_mult_eval = 1.0
best_ar_mult_eval = 1.0

best_epoch = 0
bad_epochs = 0
best_lam_ar = float(cfg.lam_ar)
best_lam_band = float(cfg.lam_band)

global_step = 0

for epoch in range(1, cfg.epochs + 1):
    t0 = time.time()
    model.train()

    # =========================
    # 1) TRAIN (mini-batches)
    # =========================
    for x_cur, x_prev, y_fut in train_loader:
        x_cur, x_prev, y_fut = x_cur.to(device), x_prev.to(device), y_fut.to(device)
        y_true = y_fut.squeeze(1)

        # warmup (contrôleurs + pénalités fortes) : laisse apprendre la recon d'abord
        warmup_done = (global_step >= int(cfg.warmup_steps))

        # ramps (évite qu'une pénalité domine d'un coup)
        if warmup_done and int(cfg.band_ramp_steps) > 0:
            band_mult = min(1.0, float(global_step - int(cfg.warmup_steps)) / float(int(cfg.band_ramp_steps)))
        else:
            band_mult = 0.0 if not warmup_done else 1.0

        if warmup_done and int(cfg.ar_ramp_steps) > 0:
            ar_mult = min(1.0, float(global_step - int(cfg.warmup_steps)) / float(int(cfg.ar_ramp_steps)))
        else:
            ar_mult = 0.0 if not warmup_done else 1.0

        lam_band_eff = float(cfg.lam_band) * float(band_mult)
        lam_ar_eff   = float(cfg.lam_ar)   * float(ar_mult)

        # ramp multipliers are kept local (do NOT mutate cfg to avoid dataclass copy issues)

        # forward
        y_hat, aux = model(x_cur)
        mse = F.mse_loss(y_hat, y_true)
        loss = mse

        # KL reg (per-unit)
        if (cfg.model != "det") and (beta_vec is not None):
            kl_bj = aux["kl_per_unit"]  # (B, n_units)
            reg_kl = (kl_bj * beta_vec.view(1, -1)).sum(dim=1).mean()
            loss = loss + reg_kl
        else:
            reg_kl = torch.tensor(0.0, device=device)

        # band penalty (per-neuron mu^2)
        if cfg.model != "det":
            lo_eff, hi_eff = current_band()

            # option "focus bas": si beaucoup d'unités sont trop basses,
            # on pénalise moins la partie haute
            wh = 1.0
            if cfg.band_focus_low and warmup_done:
                try:
                    if (reg is not None) and (reg.mu2_ema is not None):
                        frac_low_proxy = float((reg.mu2_ema < lo_eff).float().mean().item())
                        if frac_low_proxy > float(cfg.band_low_focus_thresh):
                            wh = float(cfg.lam_band_hi_mult)
                except Exception:
                    pass

            band, band_stats, _mu2_vec = hinge_band_penalty(
                aux["mu"], lo_eff, hi_eff,
                mode=cfg.band_penalty_mode,
                eps=cfg.band_rel_eps,
                weight_low=1.0,
                weight_high=wh,
                return_mu2=True
            )
            loss = loss + lam_band_eff * band
        else:
            band = torch.tensor(0.0, device=device)

        # AR penalty
        if (cfg.model != "det") and cfg.use_ar:
            _, aux_prev = model(x_prev)
            cur = aux["mu"] if cfg.ar_on_mu else aux["z"]
            prv = aux_prev["mu"] if cfg.ar_on_mu else aux_prev["z"]
            ar = ar_penalty(cur, prv, phi=cfg.phi, ar_sigma=ar_sigma)
            loss = loss + lam_ar_eff * ar
        else:
            ar = torch.tensor(0.0, device=device)

        # backward + step
        opt.zero_grad(set_to_none=True)
        loss.backward()
        if cfg.clip_grad and cfg.clip_grad > 0:
            nn.utils.clip_grad_norm_(model.parameters(), cfg.clip_grad)
        opt.step()

        # step counter (UNE seule fois)
        global_step += 1
        warmup_done = (global_step >= int(cfg.warmup_steps))

        # --- per-neuron homeostasis / thermostats (continuous monitoring, periodic act)
        if reg is not None:
            reg.observe(
                aux["mu"],
                aux["kl_per_unit"],
                aux.get("sigma", None),
                mse=mse,
                ar=ar,
                kl_w=reg_kl,
                band_w=(lam_band_eff * band)
            )

            # band auto-calibration (préserve l'hétérogénéité, ajuste l'échelle globale)
            band_calib_info = None
            if cfg.use_band_calib and warmup_done:
                try:
                    every = int(cfg.band_calib_every)
                    if (every > 0) and ((global_step % every) == 0) and (reg.mu2_ema is not None):
                        mu2e = reg.mu2_ema.detach()
                        eps0 = 1e-12

                        r_lo = (mu2e / base_lo.clamp_min(eps0)).clamp(1e-6, 1e6)
                        r_hi = (mu2e / base_hi.clamp_min(eps0)).clamp(1e-6, 1e6)

                        q_lo = float(torch.quantile(r_lo, float(cfg.band_target_low)).item())
                        q_hi = float(torch.quantile(r_hi, float(1.0 - float(cfg.band_target_high))).item())
                        # --- calibration safety: rate-limit + correct direction if band already too tight/loose
                        try:
                            lo_eff_tmp, hi_eff_tmp = current_band(lo, hi, band_state)
                            mu2e_tmp = mu2_ema.detach().float()
                            frac_low_proxy = float((mu2e_tmp < lo_eff_tmp).float().mean().item())
                            frac_high_proxy = float((mu2e_tmp > hi_eff_tmp).float().mean().item())
                        except Exception:
                            frac_low_proxy, frac_high_proxy = 0.0, 0.0

                        # If many units are below lo, lo is too strict -> nudge q_lo downward (shrink s_lo).
                        if frac_low_proxy > 0.35:
                            q_lo = min(q_lo, float(band_state.get('s_lo', 1.0)) * 0.95)
                        # If many units are above hi, hi is too strict -> nudge q_hi upward (expand s_hi).
                        if frac_high_proxy > 0.35:
                            q_hi = max(q_hi, float(band_state.get('s_hi', 1.0)) * 1.05)

                        # Rate limit per calibration step to avoid runaway (prevents sudden band blow-ups).
                        step_cap = float(getattr(cfg, 'band_calib_step_cap', 1.25))
                        s_lo_prev = float(band_state.get('s_lo', 1.0))
                        s_hi_prev = float(band_state.get('s_hi', 1.0))
                        q_lo = float(max(s_lo_prev / step_cap, min(s_lo_prev * step_cap, q_lo)))
                        q_hi = float(max(s_hi_prev / step_cap, min(s_hi_prev * step_cap, q_hi)))

                        # Absolute clamps (final safety)
                        q_lo = float(max(float(cfg.band_scale_min), min(float(cfg.band_scale_max), q_lo)))
                        q_hi = float(max(float(cfg.band_scale_min), min(float(cfg.band_scale_max), q_hi)))

                        alpha = float(cfg.band_calib_ema)

                        def _log_ema(old, new):
                            old = max(1e-6, float(old))
                            new = max(1e-6, float(new))
                            return math.exp(alpha * math.log(old) + (1.0 - alpha) * math.log(new))

                        s_lo_new = _log_ema(band_state["s_lo"], q_lo)
                        s_hi_new = _log_ema(band_state["s_hi"], q_hi)

                        s_lo_new = min(max(s_lo_new, float(cfg.band_scale_min)), float(cfg.band_scale_max))
                        s_hi_new = min(max(s_hi_new, float(cfg.band_scale_min)), float(cfg.band_scale_max))
                        s_hi_new = max(s_hi_new, s_lo_new * 1.05)

                        band_state["s_lo"] = float(s_lo_new)
                        band_state["s_hi"] = float(s_hi_new)

                        lo_eff2, hi_eff2 = current_band()
                        reg.lo = lo_eff2
                        reg.hi = hi_eff2

                        bw = float((hi_eff2 - lo_eff2).mean().item())
                        ar_sigma = ar_sigma_from_band_mu2(lo_eff2, hi_eff2, cfg.ar_k, cfg.ar_sigma_floor, cfg.ar_sigma_ceil)

                        band_calib_info = {
                            "q_lo": q_lo, "q_hi": q_hi,
                            "s_lo": float(s_lo_new), "s_hi": float(s_hi_new),
                            "band_width_mean": bw
                        }
                except Exception as e:
                    band_calib_info = {"error": str(e)}

            # periodic controller step
            if (global_step % int(cfg.ctrl_every)) == 0:
                beta_new, act_info = reg.act(cfg=cfg, warmup_done=warmup_done, ar_mult=ar_mult)
                if beta_new is not None:
                    beta_vec = beta_new
                if band_calib_info is not None:
                    try:
                        reg.last_act = {} if reg.last_act is None else reg.last_act
                        reg.last_act["band_calib"] = band_calib_info
                    except Exception:
                        pass

    # =========================
    # 2) LEGACY (only if regulator OFF) — une fois par epoch
    # =========================
    proj_info = None
    if (reg is None) and (cfg.model != "det") and cfg.use_projection and (epoch % cfg.proj_every == 0):
        lo_proj, hi_proj = current_band()
        proj_info = project_mu2_band_per_unit_on_mu_layer(
            mu_layer=model.neuron.mu,
            feature_extractor=model.feat,
            loader=train_loader,
            lo=lo_proj, hi=hi_proj,
            max_batches=cfg.proj_batches,
            max_scale=cfg.proj_max_scale,
        )

    beta_info = None
    if (reg is None) and (cfg.model != "det") and (beta_ctl is not None):
        model.eval()
        kl_list = []
        for b, (x_cur, _, _) in enumerate(train_loader):
            if b >= 10:
                break
            x_cur = x_cur.to(device)
            _, aux_tmp = model(x_cur)
            kl_list.append(aux_tmp["kl_per_unit"].detach())
        kl_bj = torch.cat(kl_list, dim=0)
        beta_vec, beta_info = beta_ctl.update(kl_bj)

    # legacy band calibration (global scaling) when regulator OFF
    band_calib_info = None
    if (reg is None) and (cfg.model != "det") and cfg.use_band_calib and warmup_done and (not cfg.use_projection):
        try:
            model.eval()
            mu2_list = []
            for b, (x_cur, _, _) in enumerate(train_loader):
                if b >= 10:
                    break
                x_cur = x_cur.to(device)
                _, aux_tmp = model(x_cur)
                mu2_list.append((aux_tmp["mu"] * aux_tmp["mu"]).mean(dim=0).detach())
            if len(mu2_list) > 0:
                mu2e = torch.stack(mu2_list, dim=0).mean(dim=0)
                eps0 = 1e-12
                r_lo = (mu2e / base_lo.clamp_min(eps0)).clamp(1e-6, 1e6)
                r_hi = (mu2e / base_hi.clamp_min(eps0)).clamp(1e-6, 1e6)
                q_lo = float(torch.quantile(r_lo, float(cfg.band_target_low)).item())
                q_hi = float(torch.quantile(r_hi, float(1.0 - float(cfg.band_target_high))).item())
                # --- calibration safety: rate-limit + correct direction if band already too tight/loose
                try:
                    lo_eff_tmp, hi_eff_tmp = current_band(lo, hi, band_state)
                    mu2e_tmp = mu2_ema.detach().float()
                    frac_low_proxy = float((mu2e_tmp < lo_eff_tmp).float().mean().item())
                    frac_high_proxy = float((mu2e_tmp > hi_eff_tmp).float().mean().item())
                except Exception:
                    frac_low_proxy, frac_high_proxy = 0.0, 0.0

                # If many units are below lo, lo is too strict -> nudge q_lo downward (shrink s_lo).
                if frac_low_proxy > 0.35:
                    q_lo = min(q_lo, float(band_state.get('s_lo', 1.0)) * 0.95)
                # If many units are above hi, hi is too strict -> nudge q_hi upward (expand s_hi).
                if frac_high_proxy > 0.35:
                    q_hi = max(q_hi, float(band_state.get('s_hi', 1.0)) * 1.05)

                # Rate limit per calibration step to avoid runaway (prevents sudden band blow-ups).
                step_cap = float(getattr(cfg, 'band_calib_step_cap', 1.25))
                s_lo_prev = float(band_state.get('s_lo', 1.0))
                s_hi_prev = float(band_state.get('s_hi', 1.0))
                q_lo = float(max(s_lo_prev / step_cap, min(s_lo_prev * step_cap, q_lo)))
                q_hi = float(max(s_hi_prev / step_cap, min(s_hi_prev * step_cap, q_hi)))

                # Absolute clamps (final safety)
                q_lo = float(max(float(cfg.band_scale_min), min(float(cfg.band_scale_max), q_lo)))
                q_hi = float(max(float(cfg.band_scale_min), min(float(cfg.band_scale_max), q_hi)))
                alpha = float(cfg.band_calib_ema)

                def _log_ema(old, new):
                    old = max(1e-6, float(old))
                    new = max(1e-6, float(new))
                    return math.exp(alpha * math.log(old) + (1.0 - alpha) * math.log(new))

                s_lo_new = _log_ema(band_state["s_lo"], q_lo)
                s_hi_new = _log_ema(band_state["s_hi"], q_hi)
                s_lo_new = min(max(s_lo_new, float(cfg.band_scale_min)), float(cfg.band_scale_max))
                s_hi_new = min(max(s_hi_new, float(cfg.band_scale_min)), float(cfg.band_scale_max))
                s_hi_new = max(s_hi_new, s_lo_new * 1.05)
                band_state["s_lo"] = float(s_lo_new)
                band_state["s_hi"] = float(s_hi_new)
                lo2, hi2 = current_band()
                bw2 = float((hi2 - lo2).mean().item())
                ar_sigma = ar_sigma_from_band_mu2(lo2, hi2, cfg.ar_k, cfg.ar_sigma_floor, cfg.ar_sigma_ceil)
                band_calib_info = {"q_lo": q_lo, "q_hi": q_hi, "s_lo": float(s_lo_new), "s_hi": float(s_hi_new), "band_width_mean": bw2}
        except Exception as e:
            band_calib_info = {"error": str(e)}

    # =========================
    # 3) EVAL + LOGS (une fois par epoch)
    # =========================


    # --- evaluation uses the same effective multipliers as training (avoid misleading shares / scores)
    warmup_done_epoch = (global_step >= int(cfg.warmup_steps))

    if warmup_done_epoch and int(cfg.band_ramp_steps) > 0:
        band_mult_eval = min(1.0, float(global_step - int(cfg.warmup_steps)) / float(int(cfg.band_ramp_steps)))
    else:
        band_mult_eval = 0.0 if not warmup_done_epoch else 1.0

    if warmup_done_epoch and int(cfg.ar_ramp_steps) > 0:
        ar_mult_eval = min(1.0, float(global_step - int(cfg.warmup_steps)) / float(int(cfg.ar_ramp_steps)))
    else:
        ar_mult_eval = 0.0 if not warmup_done_epoch else 1.0

    lam_band_eff_eval = float(cfg.lam_band) * float(band_mult_eval)
    lam_ar_eff_eval   = float(cfg.lam_ar)   * float(ar_mult_eval)

    tr = eval_epoch(train_loader, band_mult=band_mult_eval, ar_mult=ar_mult_eval)
    va = eval_epoch(val_loader, band_mult=band_mult_eval, ar_mult=ar_mult_eval)
    te = eval_epoch(test_loader, band_mult=band_mult_eval, ar_mult=ar_mult_eval)
    secs = time.time() - t0

    history.append({
        "epoch": epoch,
        "train": tr,
        "val": va,
        "test": te,
        "secs": secs,
        "proj": proj_info,
        "beta": beta_info,
        "reg_last": (None if reg is None else reg.last_act)
    })

    # selection metric
    sel = va["mse"] if cfg.select_by == "mse" else probml_proxy(va, cfg)

    if sel < (best_val - float(cfg.early_stop_min_delta)):
        best_val = sel
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        # also snapshot autopilot state so that final evaluation matches the selected checkpoint
        best_band_state = dict(band_state)
        best_beta_vec = None if (beta_vec is None) else beta_vec.detach().cpu().clone()
        best_ar_sigma = float(ar_sigma) if isinstance(ar_sigma, (float,int)) else float(getattr(ar_sigma, 'item', lambda: ar_sigma)())
        best_band_mult_eval = float(band_mult_eval)
        best_ar_mult_eval = float(ar_mult_eval)
        best_epoch = epoch
        bad_epochs = 0
        best_lam_ar = float(cfg.lam_ar)
        best_lam_band = float(cfg.lam_band)
        best_bundle = {
            "model": best_state,
            "band_state": best_band_state,
            "beta_vec": best_beta_vec,
            "ar_sigma": best_ar_sigma,
            "band_mult_eval": best_band_mult_eval,
            "ar_mult_eval": best_ar_mult_eval,
            "lam_ar": best_lam_ar,
            "lam_band": best_lam_band,
        }
    else:
        bad_epochs += 1

    # log summary
    ar_share = 0.0
    if cfg.use_ar and va["loss"] > 0:
        ar_share = 100.0 * (float(va.get('lam_ar_eff', cfg.lam_ar)) * va['ar'] / max(1e-12, va['loss']))

    msg = (f"[ep {epoch:03d}] val_mse={va['mse']:.5f} test_mse={te['mse']:.5f} "
           f"loss={va['loss']:.4f} band={va['band']:.4f} "
           f"mae={va['mae']:.5f} kl={va['kl']:.4f} mu2={va['mu2_mean']:.4f} "
           f"low={va['frac_too_low']:.3f} high={va['frac_too_high']:.3f} "
           f"ar={va['ar']:.4f} (share~{ar_share:.1f}%) lam_ar_eff={lam_ar_eff_eval:.4g} secs={secs:.1f} | band_s=({band_state['s_lo']:.3f},{band_state['s_hi']:.3f})")

    if proj_info is not None:
        msg += (f" | proj(mu2_before={proj_info.get('mu2_mean_before',0):.4f}, "
                f"s[{proj_info.get('s_min',1):.2f},{proj_info.get('s_max',1):.2f}])")

    if beta_info is not None:
        msg += (f" | beta(mean={beta_info['beta_mean']:.2e}, "
                f"range=[{beta_info['beta_min']:.2e},{beta_info['beta_max']:.2e}], "
                f"kl_mean={beta_info['kl_mean']:.3f})")
    elif (reg is not None) and (reg.last_act is not None) and ("beta" in reg.last_act):
        bi = reg.last_act["beta"]
        msg += (f" | beta(mean={bi['beta_mean']:.2e}, "
                f"range=[{bi['beta_min']:.2e},{bi['beta_max']:.2e}], "
                f"kl_mean={bi['kl_mean']:.3f})")

    if (reg is not None) and (reg.last_act is not None) and ("mu_homeostat" in reg.last_act):
        hi2 = reg.last_act["mu_homeostat"]
        msg += f" | homeo(s[{hi2.get('s_min',1):.2f},{hi2.get('s_max',1):.2f}] low={hi2.get('frac_too_low',0):.3f})"
        if "ar_share" in reg.last_act:
            msg += f" | ar_share_ema={reg.last_act['ar_share']*100:.1f}%"

    print(msg)

    if int(cfg.early_stop_patience) > 0 and bad_epochs >= int(cfg.early_stop_patience):
        print(f"[early-stop] stop at ep={epoch} (best_ep={best_epoch}, best_score={best_val:.6f}, select_by={cfg.select_by})")
        break

# restore best (model + autopilot state)
if best_bundle is not None:
    model.load_state_dict(best_bundle["model"])
    # restore cfg scalars
    cfg.lam_ar = float(best_bundle.get("lam_ar", cfg.lam_ar))
    cfg.lam_band = float(best_bundle.get("lam_band", cfg.lam_band))
    # restore band scaling
    if best_bundle.get("band_state", None) is not None:
        band_state["s_lo"] = float(best_bundle["band_state"].get("s_lo", band_state["s_lo"]))
        band_state["s_hi"] = float(best_bundle["band_state"].get("s_hi", band_state["s_hi"]))
    # recompute band + ar_sigma from restored band (preferred) unless explicitly stored
    lo_eff_best, hi_eff_best = current_band()
    ar_sigma = ar_sigma_from_band_mu2(lo_eff_best, hi_eff_best, cfg.ar_k, cfg.ar_sigma_floor, cfg.ar_sigma_ceil)
    if best_bundle.get("ar_sigma", None) is not None:
        try:
            ar_sigma = float(best_bundle["ar_sigma"])
        except Exception:
            pass
    # restore beta_vec (eval uses it)
    if best_bundle.get("beta_vec", None) is not None:
        try:
            bv = best_bundle["beta_vec"].to(device)
            beta_vec = bv
        except Exception:
            pass
    # keep regulator in sync (its lo/hi are used by homeostat)
    if reg is not None:
        reg.lo = lo_eff_best
        reg.hi = hi_eff_best

print(f"[best] epoch={best_epoch} best_score={best_val:.6f} (select_by={cfg.select_by}) lam_ar={cfg.lam_ar:.4g} lam_band={cfg.lam_band:.4g}")

# final test evaluation uses the same multipliers as the selected checkpoint (coherent reporting)
bm = 1.0 if best_bundle is None else float(best_bundle.get("band_mult_eval", 1.0))
am = 1.0 if best_bundle is None else float(best_bundle.get("ar_mult_eval", 1.0))
final_test = eval_epoch(test_loader, band_mult=bm, ar_mult=am)
final_test

[ep 001] val_mse=0.21485 test_mse=0.24445 loss=0.5167 band=0.0676 mae=0.33219 kl=98.1966 mu2=0.0754 low=0.235 high=0.103 ar=0.0000 (share~0.0%) lam_ar_eff=1.859 secs=23.2 | band_s=(1.143,1.200) | beta(mean=1.01e-03, range=[1.00e-03,1.15e-03], kl_mean=0.136) | homeo(s[0.98,1.01] low=0.000) | ar_share_ema=0.0%
[ep 002] val_mse=0.20349 test_mse=0.23074 loss=0.4500 band=0.0664 mae=0.32469 kl=47.6232 mu2=0.0812 low=0.239 high=0.071 ar=0.0000 (share~0.0%) lam_ar_eff=5.053 secs=23.5 | band_s=(1.398,1.467) | beta(mean=9.40e-04, range=[8.26e-04,1.15e-03], kl_mean=0.053) | homeo(s[0.99,1.02] low=0.000) | ar_share_ema=0.0%
[ep 003] val_mse=0.19574 test_mse=0.22401 loss=0.4041 band=0.0542 mae=0.32067 kl=47.7047 mu2=0.0862 low=0.273 high=0.065 ar=0.0000 (share~0.0%) lam_ar_eff=10 secs=22.8 | band_s=(1.598,1.678) | beta(mean=7.99e-04, range=[5.28e-04,1.15e-03], kl_mean=0.051) | homeo(s[0.98,1.02] low=0.000) | ar_share_ema=0.0%
[ep 004] val_mse=0.18816 test_mse=0.21507 loss=0.4336 band=0.0652 mae=0.3

{'loss': 0.48442381274663027,
 'mse': 0.2125864440555513,
 'mae': 0.3314094738796864,
 'kl': 76.55126389928324,
 'band': 0.06784655388874057,
 'ar': 0.0,
 'mu2_mean': 0.1423161478928361,
 'frac_too_low': 0.32357289591633465,
 'frac_too_high': 0.112296906125498,
 'band_mult': 1.0,
 'ar_mult': 1.0,
 'lam_band_eff': 3.0,
 'lam_ar_eff': 10.0}

In [17]:
#t2


In [18]:

# ============================================================
# Autopilot hyperparameter search (successive halving)
# Works with the Neuron‑VAE TS notebook objects:
#   - Cfg dataclass
#   - set_seed, make_mu2_band, hinge_band_penalty, ar_penalty, ar_sigma_from_band
#   - ModelDeterministicTS, ModelNeuronVAETS
#   - PerUnitBetaController, PerNeuronVAERegulator
# And expects you already built:
#   - train_loader, val_loader, test_loader
#   - device (torch.device)
# ============================================================

import os, json, math, time, hashlib, random
from dataclasses import asdict, replace
from typing import Dict, Any, List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.cuda.amp import autocast

# -------------------------
# Device helper (avoid _get_device(cfg) AttributeError)
# -------------------------
def _get_device(cfg=None):
    """Return a torch.device without requiring _get_device(cfg) to exist."""
    if cfg is not None:
        d = getattr(cfg, "_device", None)
        if d is not None:
            return d
    # Prefer the notebook-global device if present
    d = globals().get("device", None)
    if d is not None:
        return d
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


# --- Pull required symbols from the current notebook (__main__) when used in Jupyter
# This lets you keep all model/penalty definitions in the notebook and only import the search helper.
try:
    import __main__ as _M
    for _name in [
        "set_seed",
        "make_mu2_band",
        "project_mu2_band_per_unit_on_mu_layer",
        "hinge_band_penalty",
        "ar_penalty",
        "ar_sigma_from_band",
        "ModelDeterministicTS",
        "ModelNeuronVAETS",
        "PerUnitBetaController",
        "PerNeuronVAERegulator",
    ]:
        if _name not in globals() and hasattr(_M, _name):
            globals()[_name] = getattr(_M, _name)
except Exception:
    pass


# -------------------------
# Utils: atomic save
# -------------------------
def _atomic_write_text(path: str, text: str):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        f.write(text)
    os.replace(tmp, path)

def _atomic_torch_save(path: str, obj):
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)

def _hash_cfg(d: Dict[str, Any]) -> str:
    s = json.dumps(d, sort_keys=True, ensure_ascii=False)
    return hashlib.md5(s.encode("utf-8")).hexdigest()[:10]

# -------------------------
# Scoring: lexicographic (constraints first, then val_mse)
# -------------------------
def trial_penalty(summary: Dict[str, float], cfg) -> float:
    # penalties are "distance above target"
    low_t = float(getattr(cfg, "band_target_low", 0.20))
    high_t = float(getattr(cfg, "band_target_high", 0.10))
    ar_cap = float(getattr(cfg, "ar_share_cap", 0.30))

    p = 0.0
    p += max(0.0, float(summary.get("frac_too_low", 1.0)) - low_t) ** 2
    p += max(0.0, float(summary.get("frac_too_high", 1.0)) - high_t) ** 2
    p += max(0.0, float(summary.get("ar_share", 0.0)) - ar_cap) ** 2

    # Optional KL cap (set cfg.kl_cap if you want)
    if hasattr(cfg, "kl_cap") and cfg.kl_cap is not None:
        kl = float(summary.get("kl", 0.0))
        cap = float(cfg.kl_cap)
        p += max(0.0, kl - cap) ** 2 / (cap * cap + 1e-12)

    return float(p)

def rank_key(summary: Dict[str, float], cfg) -> Tuple[float, float]:
    return (trial_penalty(summary, cfg), float(summary.get("best_val_mse", 1e18)))

# -------------------------
# Sampling: small, high-leverage params only
# -------------------------
def sample_params(rng: np.random.RandomState) -> Dict[str, Any]:
    # log-uniform helpers
    def logu(lo, hi):
        return float(10 ** rng.uniform(math.log10(lo), math.log10(hi)))

    lr = logu(1e-4, 5e-3)
    sigma_init = logu(0.2, 1.5)
    lam_band = logu(0.8, 8.0)
    eta_mu = float(rng.uniform(0.06, 0.30))
    max_scale_step = float(rng.uniform(1.2, 2.5))
    warmup_steps = int(rng.uniform(50, 450))
    phi = float(rng.uniform(0.80, 0.98))
    ar_k = float(rng.uniform(0.4, 1.2))
    hetero_spread = float(logu(2.0, 16.0))

    # clamp beta max tighter than 1e-1 to avoid huge spikes
    beta_max = float(logu(2e-3, 3e-2))
    beta_init = float(min(1e-3, 0.5 * beta_max))

    return dict(
        lr=lr,
        sigma_init=sigma_init,
        lam_band=lam_band,
        eta_mu=eta_mu,
        max_scale_step=max_scale_step,
        warmup_steps=warmup_steps,
        phi=phi,
        ar_k=ar_k,
        hetero_spread=hetero_spread,
        beta_init=beta_init,
        beta_max=beta_max,
        # stable defaults (you can still override in base cfg)
        band_mode="hetero",
        hetero_shuffle=False,
        homeo_inside_target="center_geom",
    )

# -------------------------
# One training/eval trial (from scratch)
# -------------------------
@torch.no_grad()
def _eval_epoch(model, loader, cfg, beta_vec, current_band_fn, ar_sigma, *, use_amp: bool = False) -> Dict[str, float]:
    model.eval()
    total = 0
    mse_sum = mae_sum = loss_sum = 0.0
    kl_sum = band_sum = ar_sum = 0.0
    mu2_mean_acc = low_acc = high_acc = 0.0
    n_batches = 0

    for x_cur, x_prev, y_fut in loader:
        x_cur, x_prev, y_fut = x_cur.to(device, non_blocking=True), x_prev.to(device, non_blocking=True), y_fut.to(device, non_blocking=True)
        y_true = y_fut.squeeze(1)

        with autocast(enabled=use_amp):
            y_hat, aux = model(x_cur)
            mse = F.mse_loss(y_hat, y_true)
            mae = F.l1_loss(y_hat, y_true)

        loss = mse
        kl = aux["kl"].mean() if "kl" in aux else torch.tensor(0.0, device=_get_device(cfg))

        if (cfg.model != "det") and (beta_vec is not None):
            kl_bj = aux["kl_per_unit"]
            reg = (kl_bj * beta_vec.view(1, -1)).sum(dim=1).mean()
            loss = loss + reg

        band = torch.tensor(0.0, device=_get_device(cfg))
        stats = {"mu2_mean": 0.0, "frac_too_low": 0.0, "frac_too_high": 0.0}
        if cfg.model != "det":
            lo, hi = current_band_fn()
            band, stats = hinge_band_penalty(aux["mu"], lo, hi, mode=cfg.band_penalty_mode, eps=cfg.band_rel_eps)
            loss = loss + cfg.lam_band * band

        ar = torch.tensor(0.0, device=_get_device(cfg))
        if (cfg.model != "det") and cfg.use_ar:
            with autocast(enabled=use_amp):
                _, aux_prev = model(x_prev)
                cur = aux["mu"] if cfg.ar_on_mu else aux["z"]
            prv = aux_prev["mu"] if cfg.ar_on_mu else aux_prev["z"]
            ar = ar_penalty(cur, prv, phi=cfg.phi, ar_sigma=ar_sigma)
            loss = loss + cfg.lam_ar * ar

        B = x_cur.size(0)
        total += B
        mse_sum += float(mse.item()) * B
        mae_sum += float(mae.item()) * B
        loss_sum += float(loss.item()) * B
        kl_sum += float(kl.item()) * B
        band_sum += float(band.item()) * B
        ar_sum += float(ar.item()) * B

        mu2_mean_acc += float(stats["mu2_mean"])
        low_acc += float(stats["frac_too_low"])
        high_acc += float(stats["frac_too_high"])
        n_batches += 1

    # averages
    out = dict(
        loss=loss_sum / max(1, total),
        mse=mse_sum / max(1, total),
        mae=mae_sum / max(1, total),
        kl=kl_sum / max(1, total),
        band=band_sum / max(1, total),
        ar=ar_sum / max(1, total),
        mu2_mean=mu2_mean_acc / max(1, n_batches),
        frac_too_low=low_acc / max(1, n_batches),
        frac_too_high=high_acc / max(1, n_batches),
    )
    return out

def run_trial(
    cfg,
    seed: int,
    train_loader,
    val_loader,
    test_loader,
    out_dir: Optional[str] = None,
    prune: bool = True,
) -> Dict[str, Any]:
    """
    Train from scratch with controllers ON; return summary at best val epoch.
    """
    # local device stored on cfg for eval helper
    cfg = replace(cfg)
    set_seed(int(seed))

    # model + opt
    if cfg.model == "det":
        model = ModelDeterministicTS(cfg.input_len, cfg.hidden, cfg.n_units, cfg.horizon).to(device)
    else:
        model = ModelNeuronVAETS(cfg.input_len, cfg.hidden, cfg.n_units, cfg.horizon, cfg.sigma_mode, cfg.sigma_init).to(device)

    # Multi-GPU forward wrapper (keeps base `model` accessible for regulator/projection)
    use_dp = (device.type == "cuda") and (torch.cuda.device_count() > 1) and bool(getattr(cfg, "use_data_parallel", True))
    model_fwd = nn.DataParallel(model) if use_dp else model

    # AMP (fp16) on CUDA (T4 supports fp16)
    use_amp = (device.type == "cuda") and bool(getattr(cfg, "use_amp", True))
    _amp_device = 'cuda' if torch.cuda.is_available() else 'cpu'
    scaler = torch.amp.GradScaler(_amp_device, enabled=use_amp and (_amp_device == 'cuda'))
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    # bands
    base_lo, base_hi = make_mu2_band(
        cfg.n_units, cfg.mu2_target,
        lo_k=cfg.band_lo_k, hi_k=cfg.band_hi_k,
        mode=cfg.band_mode,
        hetero_spread=cfg.hetero_spread,
        hetero_shuffle=cfg.hetero_shuffle,
        seed=0,
        device=_get_device(cfg)
    )
    band_state = {"s_lo": 1.0, "s_hi": 1.0}

    def current_band():
        lo_eff = base_lo * float(band_state["s_lo"])
        hi_eff = base_hi * float(band_state["s_hi"])
        hi_eff = torch.maximum(hi_eff, lo_eff * 1.05)
        return lo_eff, hi_eff

    lo, hi = current_band()
    ar_sigma = ar_sigma_from_band_mu2(lo, hi, cfg.ar_k, cfg.ar_sigma_floor, cfg.ar_sigma_ceil)

    # beta + regulator
    beta_ctl = None
    beta_vec = None
    if (cfg.model != "det") and cfg.use_beta_thermostat:
        beta_ctl = PerUnitBetaController(
            n_units=cfg.n_units,
            beta_init=cfg.beta_init, beta_min=cfg.beta_min, beta_max=cfg.beta_max,
            beta_ratio_min=getattr(cfg,"beta_ratio_min",0.25), beta_ratio_max=getattr(cfg,"beta_ratio_max",4.0),
            C_min=cfg.C_min, C_max=cfg.C_max,
            lr_logbeta=cfg.lr_logbeta,
            alive_min=cfg.alive_min,
            device=_get_device(cfg)
        )
        beta_vec = beta_ctl.beta_vec()

    reg = None
    if (cfg.model != "det") and cfg.use_neuron_regulator:
        reg = PerNeuronVAERegulator(
            n_units=cfg.n_units,
            lo=lo, hi=hi,
            mu_layer=model.neuron.mu,
            neuron_layer=model.neuron,
            beta_ctl=beta_ctl,
            device=_get_device(cfg),
            eta_mu=cfg.eta_mu,
            max_scale_step=cfg.max_scale_step,
            inside_target=cfg.homeo_inside_target,
            sigma_floor=cfg.sigma_floor,
            sigma_ceil=cfg.sigma_ceil,
            sigma_kick=cfg.sigma_kick,
            alive_min=cfg.alive_min,
            adapt_lam_ar=cfg.adapt_lam_ar,
            ar_share_target=cfg.ar_share_target,
            ar_lr=cfg.ar_lr,
            lam_ar_min=cfg.lam_ar_min,
            lam_ar_max=cfg.lam_ar_max,
        )
        # sync reg band
        reg.lo, reg.hi = lo, hi

    history = []
    best_score = 1e18
    best_val_mse = 1e18
    best_epoch = 0
    best_state = None
    best_metrics = None

    global_step = 0
    t_start = time.time()

    for epoch in range(1, int(cfg.epochs) + 1):
        model_fwd.train()
        for x_cur, x_prev, y_fut in train_loader:
            x_cur, x_prev, y_fut = x_cur.to(device, non_blocking=True), x_prev.to(device, non_blocking=True), y_fut.to(device, non_blocking=True)
            y_true = y_fut.squeeze(1)

            warmup_done = (global_step >= int(cfg.warmup_steps))

            # ramps
            if warmup_done and int(cfg.band_ramp_steps) > 0:
                band_mult = min(1.0, float(global_step - int(cfg.warmup_steps)) / float(int(cfg.band_ramp_steps)))
            else:
                band_mult = 0.0 if not warmup_done else 1.0

            if warmup_done and int(cfg.ar_ramp_steps) > 0:
                ar_mult = min(1.0, float(global_step - int(cfg.warmup_steps)) / float(int(cfg.ar_ramp_steps)))
            else:
                ar_mult = 0.0 if not warmup_done else 1.0

            lam_band_eff = float(cfg.lam_band) * float(band_mult)
            lam_ar_eff = float(cfg.lam_ar) * float(ar_mult)


            y_hat, aux = model_fwd(x_cur)
            mse = F.mse_loss(y_hat, y_true)
            loss = mse

            # KL weighted
            reg_kl = torch.tensor(0.0, device=_get_device(cfg))
            if (cfg.model != "det") and (beta_vec is not None):
                kl_bj = aux["kl_per_unit"]
                reg_kl = (kl_bj * beta_vec.view(1, -1)).sum(dim=1).mean()
                loss = loss + reg_kl

            # band penalty with focus-low
            band = torch.tensor(0.0, device=_get_device(cfg))
            band_stats = {"mu2_mean": 0.0, "frac_too_low": 0.0, "frac_too_high": 0.0}
            if cfg.model != "det":
                lo_eff, hi_eff = current_band()
                wh = 1.0
                if cfg.band_focus_low and (reg is not None) and (reg.mu2_ema is not None):
                    frac_low_proxy = float((reg.mu2_ema < lo_eff).float().mean().item())
                    if frac_low_proxy > float(cfg.band_low_focus_thresh):
                        wh = float(cfg.lam_band_hi_mult)

                band, band_stats, _mu2_vec = hinge_band_penalty(
                    aux["mu"], lo_eff, hi_eff,
                    mode=cfg.band_penalty_mode,
                    eps=cfg.band_rel_eps,
                    weight_low=1.0,
                    weight_high=wh,
                    return_mu2=True,
                )
                loss = loss + lam_band_eff * band

            # AR
            ar = torch.tensor(0.0, device=_get_device(cfg))
            if (cfg.model != "det") and cfg.use_ar:
                _, aux_prev = model_fwd(x_prev)
                cur = aux["mu"] if cfg.ar_on_mu else aux["z"]
                prv = aux_prev["mu"] if cfg.ar_on_mu else aux_prev["z"]
                ar = ar_penalty(cur, prv, phi=cfg.phi, ar_sigma=ar_sigma)
                loss = loss + lam_ar_eff * ar

            opt.zero_grad(set_to_none=True)
            loss.backward()
            if cfg.clip_grad and cfg.clip_grad > 0:
                nn.utils.clip_grad_norm_(model.parameters(), cfg.clip_grad)
            opt.step()

            # regulator observe + act + band calib
            if reg is not None:
                reg.observe(
                    aux["mu"],
                    aux["kl_per_unit"],
                    aux.get("sigma", None),
                    mse=mse,
                    ar=ar,
                    kl_w=reg_kl,
                    band_w=(lam_band_eff * band),
                )

                # band calibration (global scaling)
                if cfg.use_band_calib and warmup_done:
                    every = int(cfg.band_calib_every)
                    if (every > 0) and ((global_step % every) == 0) and (reg.mu2_ema is not None):
                        mu2e = reg.mu2_ema.detach()
                        eps0 = 1e-12
                        r_lo = (mu2e / base_lo.clamp_min(eps0)).clamp(1e-6, 1e6)
                        r_hi = (mu2e / base_hi.clamp_min(eps0)).clamp(1e-6, 1e6)
                        q_lo = float(torch.quantile(r_lo, float(cfg.band_target_low)).item())
                        q_hi = float(torch.quantile(r_hi, float(1.0 - float(cfg.band_target_high))).item())
                        alpha = float(cfg.band_calib_ema)

                        def _log_ema(old, new):
                            old = max(1e-6, float(old))
                            new = max(1e-6, float(new))
                            return math.exp(alpha * math.log(old) + (1.0 - alpha) * math.log(new))

                        s_lo_new = _log_ema(band_state["s_lo"], q_lo)
                        s_hi_new = _log_ema(band_state["s_hi"], q_hi)
                        s_lo_new = min(max(s_lo_new, float(cfg.band_scale_min)), float(cfg.band_scale_max))
                        s_hi_new = min(max(s_hi_new, float(cfg.band_scale_min)), float(cfg.band_scale_max))
                        s_hi_new = max(s_hi_new, s_lo_new * 1.05)
                        band_state["s_lo"] = float(s_lo_new)
                        band_state["s_hi"] = float(s_hi_new)

                        lo2, hi2 = current_band()
                        reg.lo = lo2
                        reg.hi = hi2
                        bw = float((hi2 - lo2).mean().item())
                        ar_sigma = ar_sigma_from_band(bw, cfg.ar_k, cfg.ar_sigma_floor, cfg.ar_sigma_ceil)

                # controller step
                if (global_step % int(cfg.ctrl_every)) == 0:
                    beta_new, _info = reg.act(cfg=cfg, warmup_done=warmup_done, ar_mult=ar_mult)
                    if beta_new is not None:
                        beta_vec = beta_new

            global_step += 1


        # end epoch: optional legacy controls when regulator OFF (ne casse rien)
        warmup_done = (global_step >= int(cfg.warmup_steps))

        beta_info = None
        if (reg is None) and (cfg.model != "det") and (beta_ctl is not None):
            model_fwd.eval()
            kl_list = []
            for b, (x_cur, _, _) in enumerate(train_loader):
                if b >= 10:
                    break
                x_cur = x_cur.to(device)
                _, aux_tmp = model(x_cur)
                kl_list.append(aux_tmp["kl_per_unit"].detach())
            if len(kl_list) > 0:
                kl_bj = torch.cat(kl_list, dim=0)
                beta_vec, beta_info = beta_ctl.update(kl_bj)

        proj_info = None
        if (reg is None) and (cfg.model != "det") and bool(getattr(cfg, "use_projection", False)):
            every = int(getattr(cfg, "proj_every", 1))
            if every > 0 and (epoch % every == 0):
                lo_proj, hi_proj = current_band()
                proj_info = project_mu2_band_per_unit_on_mu_layer(
                    mu_layer=model.neuron.mu,
                    feature_extractor=model.feat,
                    loader=train_loader,
                    lo=lo_proj, hi=hi_proj,
                    max_batches=int(getattr(cfg, "proj_batches", 50)),
                    max_scale=float(getattr(cfg, "proj_max_scale", 50.0)),
                )

        band_calib_info = None
        if (reg is None) and (cfg.model != "det") and bool(getattr(cfg, "use_band_calib", False)) and warmup_done and (not bool(getattr(cfg, "use_projection", False))):
            try:
                # calibration douce: ajuste l'échelle globale des bandes via quantiles
                model_fwd.eval()
                mu2_list = []
                for b, (x_cur, _, _) in enumerate(train_loader):
                    if b >= 10:
                        break
                    x_cur = x_cur.to(device)
                    _, aux_tmp = model(x_cur)
                    mu2_list.append((aux_tmp["mu"] * aux_tmp["mu"]).mean(dim=0).detach())
                if len(mu2_list) > 0:
                    mu2e = torch.stack(mu2_list, dim=0).mean(dim=0)
                    eps0 = 1e-12
                    r_lo = (mu2e / base_lo.clamp_min(eps0)).clamp(1e-6, 1e6)
                    r_hi = (mu2e / base_hi.clamp_min(eps0)).clamp(1e-6, 1e6)

                    q_lo = float(torch.quantile(r_lo, float(cfg.band_target_low)).item())
                    q_hi = float(torch.quantile(r_hi, float(1.0 - float(cfg.band_target_high))).item())
                    alpha = float(cfg.band_calib_ema)

                    def _log_ema(old, new):
                        old = max(1e-6, float(old))
                        new = max(1e-6, float(new))
                        return math.exp(alpha * math.log(old) + (1.0 - alpha) * math.log(new))

                    s_lo_new = _log_ema(band_state["s_lo"], q_lo)
                    s_hi_new = _log_ema(band_state["s_hi"], q_hi)
                    s_lo_new = min(max(s_lo_new, float(cfg.band_scale_min)), float(cfg.band_scale_max))
                    s_hi_new = min(max(s_hi_new, float(cfg.band_scale_min)), float(cfg.band_scale_max))
                    s_hi_new = max(s_hi_new, s_lo_new * 1.05)

                    band_state["s_lo"] = float(s_lo_new)
                    band_state["s_hi"] = float(s_hi_new)

                    lo2, hi2 = current_band()
                    ar_sigma = ar_sigma_from_band_mu2(lo2, hi2, cfg.ar_k, cfg.ar_sigma_floor, cfg.ar_sigma_ceil)

                    band_calib_info = {"q_lo": q_lo, "q_hi": q_hi, "s_lo": float(s_lo_new), "s_hi": float(s_hi_new), "band_width_mean": bw2}
            except Exception as e:
                band_calib_info = {"error": str(e)}

        # end epoch: eval
        val = _eval_epoch(model_fwd, val_loader, cfg, beta_vec, current_band, ar_sigma, use_amp=use_amp)
        test = _eval_epoch(model_fwd, test_loader, cfg, beta_vec, current_band, ar_sigma, use_amp=use_amp)
        act = {} if (reg is None or reg.last_act is None) else dict(reg.last_act)
        # attach legacy infos for reporting (paper-friendly)
        if beta_info is not None:
            act["beta"] = beta_info
        if proj_info is not None:
            act["proj"] = proj_info
        if band_calib_info is not None:
            act["band_calib"] = band_calib_info
        ar_adapt_info = None
        if (reg is None) and (cfg.model != "det") and cfg.use_ar and cfg.adapt_lam_ar:
            denom = max(1e-12, float(val.get("loss", 0.0)))
            share = (float(cfg.lam_ar) * float(val.get("ar", 0.0))) / denom

            target = float(cfg.ar_share_target)
            cap = getattr(cfg, "ar_share_cap", None)
            cap = None if cap is None else float(cap)

            err = share - target
            lam = float(cfg.lam_ar)
            new_lam = lam * math.exp(-float(cfg.ar_lr) * err)
            if (cap is not None) and (share > cap):
                new_lam = new_lam * math.exp(-2.0 * float(cfg.ar_lr) * (share - cap))
            new_lam = min(max(new_lam, float(cfg.lam_ar_min)), float(cfg.lam_ar_max))

            ar_adapt_info = {"ar_share": float(share), "ar_share_target": float(target),
                             "lam_ar_old": float(lam), "lam_ar_new": float(new_lam)}
            if cap is not None:
                ar_adapt_info["ar_share_cap"] = float(cap)

            cfg.lam_ar = float(new_lam)

        if ar_adapt_info is not None:
            act["ar_adapt"] = ar_adapt_info
            act["ar_share"] = float(ar_adapt_info.get("ar_share", 0.0))

        row = dict(epoch=epoch, **{f"val_{k}": v for k, v in val.items()}, **{f"test_{k}": v for k, v in test.items()})
        if "ar_share" in act:
            row["ar_share"] = float(act["ar_share"])
        history.append(row)

        # pruning (cheap): if hopeless, abort early
        if prune and epoch >= 2:
            if float(val["mse"]) > 5.0:   # sanity: exploding
                break
            if (not bool(getattr(cfg, "use_projection", False))) and float(val.get("frac_too_low", 1.0)) > 0.98 and epoch >= 3:
                break

        sel_by = str(getattr(cfg, "select_by", "mse")).lower().strip()
        sel_score = float(val.get("loss", val["mse"])) if sel_by in ("probml", "loss") else float(val["mse"])
        if float(sel_score) + 1e-12 < best_score:
            best_score = float(sel_score)
            best_val_mse = float(val["mse"])
            best_epoch = int(epoch)
            best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
            best_metrics = dict(val=val, test=test, act=act)

    secs = time.time() - t_start

    summary = {
        "seed": int(seed),
        "best_epoch": int(best_epoch),
        "best_val_mse": float(best_val_mse),
        "best_score": float(best_score),
        "select_by": str(getattr(cfg, "select_by", "mse")),
        "secs": float(secs),
    }
    if best_metrics is not None:
        summary.update(best_metrics["val"])
        # keep constraint metrics at best
        summary["frac_too_low"] = float(best_metrics["val"]["frac_too_low"])
        summary["frac_too_high"] = float(best_metrics["val"]["frac_too_high"])
        summary["kl"] = float(best_metrics["val"]["kl"])
        summary["mu2_mean"] = float(best_metrics["val"]["mu2_mean"])
        if "ar_share" in best_metrics["act"]:
            summary["ar_share"] = float(best_metrics["act"]["ar_share"])
        summary["lam_ar_final"] = float(getattr(cfg, "lam_ar", 0.0))
        if best_metrics["act"].get("beta") is not None:
            b = best_metrics["act"]["beta"]
            try:
                summary["beta_mean"] = float(b.get("mean", np.nan))
                summary["beta_min"] = float(b.get("min", np.nan))
                summary["beta_max"] = float(b.get("max", np.nan))
                summary["kl_mean_unit"] = float(b.get("kl_mean", np.nan))
            except Exception:
                pass

    # Optional saving
    if out_dir is not None:
        os.makedirs(out_dir, exist_ok=True)
        _atomic_write_text(os.path.join(out_dir, "cfg.json"), json.dumps(asdict(cfg), indent=2, ensure_ascii=False))
        _atomic_write_text(os.path.join(out_dir, "summary.json"), json.dumps(summary, indent=2, ensure_ascii=False))
        _atomic_write_text(os.path.join(out_dir, "history.json"), json.dumps(history, indent=2, ensure_ascii=False))
        # Save torch checkpoints only if explicitly enabled (keeps Kaggle storage small)
        if bool(getattr(cfg, "save_pt", False)) and best_state is not None:
            _atomic_torch_save(os.path.join(out_dir, "best_state.pt"), best_state)

    return {"summary": summary, "history": history}

# -------------------------
# Successive halving
# -------------------------
def successive_halving_search(
    base_cfg,
    train_loader,
    val_loader,
    test_loader,
    out_root: str = "./autopilot_runs",
    n_trials: int = 24,
    budgets: List[int] = [3, 6, 10],
    keep_frac: float = 0.35,
    seed_stage: int = 0,
    seeds_final: List[int] = [0, 100, 200],
    rng_seed: int = 0,
) -> Dict[str, Any]:
    os.makedirs(out_root, exist_ok=True)
    device = globals().get('device', torch.device('cpu'))
    rng = np.random.RandomState(int(rng_seed))

    # initial pool
    pool = []
    for t in range(int(n_trials)):
        params = sample_params(rng)
        cfg_t = replace(base_cfg, **params)
        pool.append({"cfg": cfg_t, "params": params, "id": t})

    all_rows = []

    for si, epochs in enumerate(budgets):
        stage_dir = os.path.join(out_root, f"stage_{si+1}_ep{int(epochs)}")
        os.makedirs(stage_dir, exist_ok=True)

        # choose seeds: cheap early, robust final
        seeds = [int(seed_stage)] if si < (len(budgets) - 1) else list(seeds_final)

        scored = []
        for item in pool:
            cfg_t = replace(item["cfg"], epochs=int(epochs))
            cfg_dict = asdict(cfg_t)
            h = _hash_cfg(cfg_dict)
            trial_dir = os.path.join(stage_dir, f"trial_{item['id']:04d}_{h}")

            # run (multi-seed aggregate)
            summaries = []
            for sd in seeds:
                out = run_trial(cfg_t, seed=sd, train_loader=train_loader, val_loader=val_loader, test_loader=test_loader,
                                out_dir=os.path.join(trial_dir, f"seed_{sd}"), prune=True)
                summaries.append(out["summary"])

            # aggregate (mean over seeds)
            def _mean(key, default=np.nan):
                xs = [float(s.get(key, default)) for s in summaries]
                xs = [x for x in xs if np.isfinite(x)]
                return float(np.mean(xs)) if len(xs) else float(default)

            agg = {
                "stage": si + 1,
                "epochs": int(epochs),
                "trial_id": item["id"],
                "hash": h,
                "best_val_mse": _mean("best_val_mse"),
                "frac_too_low": _mean("frac_too_low"),
                "frac_too_high": _mean("frac_too_high"),
                "kl": _mean("kl"),
                "mu2_mean": _mean("mu2_mean"),
                "ar_share": _mean("ar_share", default=0.0),
                "secs": _mean("secs"),
            }
            # store config
            _atomic_write_text(os.path.join(trial_dir, "cfg_compact.json"), json.dumps(cfg_dict, ensure_ascii=False))
            _atomic_write_text(os.path.join(trial_dir, "agg.json"), json.dumps(agg, indent=2, ensure_ascii=False))

            all_rows.append({**agg, **item["params"]})
            scored.append((rank_key(agg, cfg_t), agg, item, cfg_t))

        # keep best fraction
        scored.sort(key=lambda x: x[0])
        k = max(1, int(math.ceil(len(scored) * float(keep_frac))))
        pool = [{"cfg": s[3], "params": s[2]["params"], "id": s[2]["id"]} for s in scored[:k]]

        # write stage table
        _atomic_write_text(os.path.join(stage_dir, "leaderboard.json"),
                           json.dumps([s[1] for s in scored], indent=2, ensure_ascii=False))

    # overall best from last stage
    # rebuild leaderboard from last stage rows
    last_rows = [r for r in all_rows if r["stage"] == len(budgets)]
    last_rows.sort(key=lambda r: rank_key(r, base_cfg))
    best = last_rows[0] if last_rows else None

    _atomic_write_text(os.path.join(out_root, "results.json"), json.dumps(all_rows, indent=2, ensure_ascii=False))
    _atomic_write_text(os.path.join(out_root, "best.json"), json.dumps(best, indent=2, ensure_ascii=False))

    return {"best": best, "all": all_rows}


In [19]:

# --- Run search
base_cfg = cfg  # your Cfg() instance

out = successive_halving_search(
    base_cfg,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    out_root='./autopilot_runs',
    n_trials=24,
    budgets=[3,6,10],
    keep_frac=0.35,
    seed_stage=0,
    seeds_final=[0,100,200],
    rng_seed=0,
)

print('BEST:', out['best'])


/tmp/ipykernel_55/3029606597.py:161: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


BEST: {'stage': 3, 'epochs': 10, 'trial_id': 22, 'hash': 'f20bc2fbd6', 'best_val_mse': 0.1970506730406455, 'frac_too_low': 0.3106778614707835, 'frac_too_high': 0.07715103129150067, 'kl': 63.7784813343971, 'mu2_mean': 0.11797923155890834, 'ar_share': 0.0, 'secs': 172.9929071267446, 'lr': 0.0011711736306190274, 'sigma_init': 1.160520404522885, 'lam_band': 1.5018688916629197, 'eta_mu': 0.2515312401390153, 'max_scale_step': 1.4413267275977377, 'warmup_steps': 431, 'phi': 0.9237478897498067, 'ar_k': 0.5724061416908468, 'hetero_spread': 14.341378235113982, 'beta_init': 0.001, 'beta_max': 0.014473818477155712, 'band_mode': 'hetero', 'hetero_shuffle': False, 'homeo_inside_target': 'center_geom'}


In [20]:
# (4) Load LongHorizon2 datasets (ECL / TrafficL / Exchange / Weather)
from datasetsforecast.long_horizon2 import LongHorizon2

def load_longhorizon_group(group: str, data_dir: str = "./data_longhorizon2", normalize: bool = True) -> pd.DataFrame:
    os.makedirs(data_dir, exist_ok=True)
    df = LongHorizon2.load(directory=data_dir, group=group, normalize=normalize)
    # columns: unique_id, ds, y
    return df


# Keep the same API your suite expects
def _group_alias(g: str) -> str:
    resolved, _ = _resolve_group(g, prefer="lh2")
    return resolved

In [21]:
# ============================================================
# Robust loader: LH2 if available, else LH1 (fixes Exchange)
# Put this cell BEFORE build_loaders/run_experiment/suite loop
# ============================================================

import os
import pandas as pd

def _lh2_groups():
    from datasetsforecast.long_horizon2 import LongHorizon2Info
    return list(LongHorizon2Info.groups)

def _lh1_groups():
    from datasetsforecast.long_horizon import LongHorizonInfo
    return list(LongHorizonInfo.groups)

def _group_alias(g: str) -> str:
    g = str(g).strip()
    if g.lower() == "traffic":
        return "TrafficL"
    return g

def load_longhorizon_group(group: str,
                           data_dir_lh2: str = "./data_longhorizon2",
                           data_dir_lh1: str = "./data_longhorizon",
                           normalize_lh2: bool = True,
                           cache_lh1: bool = True) -> pd.DataFrame:
    """
    Retourne toujours un DataFrame cible avec colonnes ['unique_id','ds','y'].

    - Si group ∈ LH2: LongHorizon2.load(..., normalize=...) -> DataFrame
    - Sinon si group ∈ LH1: LongHorizon.load(..., cache=...) -> (Y_df, X_df, S_df)
      -> on renvoie Y_df
    """
    group = _group_alias(group)

    g2 = set(_lh2_groups())
    g1 = set(_lh1_groups())

    if group in g2:
        from datasetsforecast.long_horizon2 import LongHorizon2
        os.makedirs(data_dir_lh2, exist_ok=True)
        df = LongHorizon2.load(directory=data_dir_lh2, group=group, normalize=normalize_lh2)
        return df

    if group in g1:
        from datasetsforecast.long_horizon import LongHorizon
        os.makedirs(data_dir_lh1, exist_ok=True)
        res = LongHorizon.load(directory=data_dir_lh1, group=group, cache=cache_lh1)
        Y_df = res[0] if isinstance(res, tuple) else res
        return Y_df

    raise ValueError(
        f"Group '{group}' introuvable.\n"
        f"LH2: {_lh2_groups()}\n"
        f"LH1: {_lh1_groups()}"
    )

# (Optionnel) mini check lisible
print("[LH2 groups]", _lh2_groups())
print("[LH1 groups]", _lh1_groups())
for g in ["ECL", "Traffic", "Exchange", "Weather"]:
    gg = _group_alias(g)
    backend = "LH2" if gg in set(_lh2_groups()) else ("LH1" if gg in set(_lh1_groups()) else "NONE")
    print(f"[route] {g:>8} -> {gg:<8} via {backend}")


[LH2 groups] ['ETTh1', 'ETTh2', 'ETTm1', 'ETTm2', 'ECL', 'TrafficL', 'Weather']
[LH1 groups] ['ETTh1', 'ETTh2', 'ETTm1', 'ETTm2', 'ECL', 'Exchange', 'TrafficL', 'ILI', 'Weather']
[route]      ECL -> ECL      via LH2
[route]  Traffic -> TrafficL via LH2
[route] Exchange -> Exchange via LH1
[route]  Weather -> Weather  via LH2


In [22]:
# (5) Windowed dataset with (x_cur, x_prev, y_future) for AR
class LongHorizonWindowDataset(Dataset):
    # For each (sid, t):
    #   x_cur  = y[t-input_len : t]                    shape (1, input_len)
    #   x_prev = y[t-input_len-stride_ar : t-stride_ar]
    #   y_fut  = y[t : t+horizon]                      shape (1, horizon)
    #
    # stride_ar is ONLY for the AR term.
    def __init__(
        self,
        df: pd.DataFrame,
        input_len: int,
        horizon: int,
        stride_ar: int,
        split: str = "train",
        train_frac: float = 0.7,
        val_frac: float = 0.1,
        max_per_series: Optional[int] = None,
        seed: int = 0,
    ):
        assert split in ("train", "val", "test")
        self.input_len = int(input_len)
        self.horizon = int(horizon)
        self.stride_ar = int(stride_ar)

        self.series = []
        for uid, g in df.groupby("unique_id"):
            y = g.sort_values("ds")["y"].to_numpy(dtype=np.float32)
            self.series.append(y)

        rng = np.random.RandomState(int(seed))

        self.indices = []
        for sid, y in enumerate(self.series):
            T = len(y)
            t_min = self.input_len + self.stride_ar
            t_max = T - self.horizon
            if t_max <= t_min:
                continue

            n = t_max - t_min
            i_train = t_min + int(train_frac * n)
            i_val   = t_min + int((train_frac + val_frac) * n)

            if split == "train":
                ts = list(range(t_min, i_train))
            elif split == "val":
                ts = list(range(i_train, i_val))
            else:
                ts = list(range(i_val, t_max))

            if max_per_series is not None and len(ts) > int(max_per_series):
                ts = list(rng.choice(ts, size=int(max_per_series), replace=False))

            self.indices.extend([(sid, t) for t in ts])

        if len(self.indices) == 0:
            raise RuntimeError("No samples: check input_len/horizon/stride_ar for this dataset.")

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx: int):
        sid, t = self.indices[idx]
        y = self.series[sid]
        L, S, H = self.input_len, self.stride_ar, self.horizon

        x_cur  = y[t-L : t]
        x_prev = y[t-L-S : t-S]
        y_fut  = y[t : t+H]

        x_cur  = torch.from_numpy(x_cur).unsqueeze(0)   # (1,L)
        x_prev = torch.from_numpy(x_prev).unsqueeze(0)  # (1,L)
        y_fut  = torch.from_numpy(y_fut).unsqueeze(0)   # (1,H)
        return x_cur, x_prev, y_fut

In [23]:
def startup_smoke_test(cfg=None, debug=True, max_groups=5):
    """
    Test rapide AU DÉMARRAGE, sans dépendre de SUITE_GROUPS.
    - charge quelques groupes disponibles
    - vérifie format minimal du DataFrame
    - si cfg est fourni, vérifie aussi longueur >= input_len+horizon
    """
    # récupère les groupes dispo
    try:
        from datasetsforecast.long_horizon2 import LongHorizon2Info
        lh2 = list(LongHorizon2Info.groups)
    except Exception:
        lh2 = []
    try:
        from datasetsforecast.long_horizon import LongHorizonInfo
        lh1 = list(LongHorizonInfo.groups)
    except Exception:
        lh1 = []

    all_groups = list(dict.fromkeys(lh2 + lh1))  # union stable
    if debug:
        print(f"[startup] LH2 groups ({len(lh2)}): {lh2}")
        print(f"[startup] LH1 groups ({len(lh1)}): {lh1}")

    # groupes "cibles" si dispo (ne crash pas si absents)
    preferred = ["ECL", "Traffic", "Exchange", "Weather", "ILI"]
    to_test = []
    s_all = set(all_groups)

    for g in preferred:
        ga = _group_alias(g)
        if ga in s_all:
            to_test.append(ga)

    # fallback: si aucun des préférés n'existe, on teste un petit échantillon
    if not to_test:
        to_test = all_groups[:max_groups]
    else:
        # limite si tu veux éviter trop de chargements au boot
        to_test = to_test[:max_groups]

    if debug:
        print(f"[startup] Testing groups: {to_test}")

    # besoin minimal si cfg est fourni
    need = None
    if cfg is not None and hasattr(cfg, "input_len") and hasattr(cfg, "horizon"):
        need = int(cfg.input_len) + int(cfg.horizon)
        if debug:
            print(f"[startup] need_len=input_len+horizon={need}")

    # test de chargement + check minimal
    for g in to_test:
        if debug:
            print(f"\n[startup] load_longhorizon_group('{g}') ...")

        df = load_longhorizon_group(g)

        # check colonnes et non-vide
        if not isinstance(df, pd.DataFrame) or len(df) == 0:
            raise RuntimeError(f"[startup] BAD df for group={g}: empty or not a DataFrame.")

        for col in ["unique_id", "ds", "y"]:
            if col not in df.columns:
                raise RuntimeError(f"[startup] BAD df for group={g}: missing column '{col}'. cols={list(df.columns)}")

        # check types rapides (sans tout recalculer)
        # ds convertible en datetime, y convertible float (coerce)
        ds_ok = pd.to_datetime(df["ds"].head(50), errors="coerce").notna().all()
        if not ds_ok:
            raise RuntimeError(f"[startup] BAD df for group={g}: 'ds' not datetime-convertible.")
        y_ok = pd.to_numeric(df["y"].head(50), errors="coerce").notna().all()
        if not y_ok:
            raise RuntimeError(f"[startup] BAD df for group={g}: 'y' not numeric-convertible.")

        # check longueur séries si possible
        if need is not None:
            sizes = df.groupby("unique_id", sort=False).size()
            if int(sizes.max()) < need:
                raise RuntimeError(
                    f"[startup] BAD df for group={g}: max series length {int(sizes.max())} < need {need}."
                )
            if debug:
                print(f"[startup] ok: rows={len(df)} series={df['unique_id'].nunique()} size[min/max]={int(sizes.min())}/{int(sizes.max())}")

    print("\n✅ [startup] smoke test OK — charge/format/longueurs semblent safe.")
    
startup_smoke_test(cfg if "cfg" in globals() else None, debug=True, max_groups=4)


[startup] LH2 groups (7): ['ETTh1', 'ETTh2', 'ETTm1', 'ETTm2', 'ECL', 'TrafficL', 'Weather']
[startup] LH1 groups (9): ['ETTh1', 'ETTh2', 'ETTm1', 'ETTm2', 'ECL', 'Exchange', 'TrafficL', 'ILI', 'Weather']
[startup] Testing groups: ['ECL', 'TrafficL', 'Exchange', 'Weather']
[startup] need_len=input_len+horizon=432

[startup] load_longhorizon_group('ECL') ...
[startup] ok: rows=8443584 series=321 size[min/max]=26304/26304

[startup] load_longhorizon_group('TrafficL') ...
[startup] ok: rows=15122928 series=862 size[min/max]=17544/17544

[startup] load_longhorizon_group('Exchange') ...


100%|██████████| 314M/314M [00:06<00:00, 45.3MiB/s] 
INFO:datasetsforecast.utils:Successfully downloaded datasets.zip, 314116557, bytes.
INFO:datasetsforecast.utils:Decompressing zip file...
INFO:datasetsforecast.utils:Successfully decompressed data_longhorizon/longhorizon/datasets/datasets.zip


[startup] ok: rows=60704 series=8 size[min/max]=7588/7588

[startup] load_longhorizon_group('Weather') ...
[startup] ok: rows=1106595 series=21 size[min/max]=52695/52695

✅ [startup] smoke test OK — charge/format/longueurs semblent safe.


In [24]:
# ============================================================
# (16c) Suite multi-datasets + ablation projection ON vs OFF (paper-friendly)
#  - écrit un Auto-Pilot report par run (JSON/CSV uniquement)
#  - agrège un tableau final (group x variant) + mean±std multi-seed
#  - IMPORTANT Kaggle: on *affiche* systématiquement les tableaux utiles (si zip manquant)
# ============================================================
from dataclasses import replace, asdict
import json, os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader

try:
    from IPython.display import display
except Exception:
    display = print  # fallback

def cfg_copy(cfg, **kwargs):
    "Copy-friendly helper (dataclass replace)."
    return replace(cfg, **kwargs)

def _group_alias(g: str) -> str:
    g = str(g)
    if g.lower() == "traffic":
        return "TrafficL"
    return g

def build_loaders(group: str, cfg, seed: int = 0):
    "Build train/val/test loaders for a given group and config."
    group = _group_alias(group)
    df = load_longhorizon_group(group)

    # Match LongHorizonWindowDataset signature defined earlier in the notebook:
    #   (df, input_len, horizon, stride_ar, split, train_frac, val_frac, max_per_series, seed)
    stride_ar = int(getattr(cfg, "stride_ar", 1))
    train_frac = float(getattr(cfg, "train_frac", 0.7))
    val_frac   = float(getattr(cfg, "val_frac", 0.2))  # (train=0.7, val=0.2, test=0.1)

    train_ds = LongHorizonWindowDataset(
        df, int(cfg.input_len), int(cfg.horizon), stride_ar,
        split="train", train_frac=train_frac, val_frac=val_frac,
        max_per_series=400, seed=int(seed),
    )
    val_ds = LongHorizonWindowDataset(
        df, int(cfg.input_len), int(cfg.horizon), stride_ar,
        split="val", train_frac=train_frac, val_frac=val_frac,
        max_per_series=200, seed=int(seed) + 1,
    )
    test_ds = LongHorizonWindowDataset(
        df, int(cfg.input_len), int(cfg.horizon), stride_ar,
        split="test", train_frac=train_frac, val_frac=val_frac,
        max_per_series=200, seed=int(seed) + 2,
    )

    train_loader = DataLoader(train_ds, batch_size=int(cfg.batch_size), shuffle=True, drop_last=True,
                              num_workers=0, pin_memory=torch.cuda.is_available())
    val_loader   = DataLoader(val_ds,   batch_size=int(cfg.batch_size), shuffle=False, drop_last=False,
                              num_workers=0, pin_memory=torch.cuda.is_available())
    test_loader  = DataLoader(test_ds,  batch_size=int(cfg.batch_size), shuffle=False, drop_last=False,
                              num_workers=0, pin_memory=torch.cuda.is_available())
    return train_loader, val_loader, test_loader

def autopilot_report(cfg, history, best_epoch, best_val, final_test,
                     run_name: str, out_root="autopilot_reports",
                     run_id: str = None, variant_name: str = None,
                     show_plots: bool = False):
    '''
    Exporte un mini-report par run (JSON/CSV uniquement) :
      - cfg.json
      - history.csv (par epoch)
      - summary.json (best-by-mse vs best-by-probml-proxy)
    NOTE: on ne sauvegarde pas de PNG/PDF (Kaggle storage). Si besoin, show_plots=True affiche inline.
    '''
    out_dir = os.path.join(out_root, (run_id or run_name))
    os.makedirs(out_dir, exist_ok=True)

    df_epochs = pd.DataFrame(history).copy()
    if "epoch" not in df_epochs.columns:
        df_epochs.insert(0, "epoch", np.arange(1, len(df_epochs) + 1))

    best_by_mse = None
    if "val_mse" in df_epochs.columns and len(df_epochs) > 0:
        i = int(df_epochs["val_mse"].astype(float).idxmin())
        best_by_mse = df_epochs.loc[i].to_dict()

    best_by_probml = None
    if "val_loss" in df_epochs.columns and len(df_epochs) > 0:
        i = int(df_epochs["val_loss"].astype(float).idxmin())
        best_by_probml = df_epochs.loc[i].to_dict()

    vname = variant_name if variant_name is not None else str(run_name.split("_")[-1])

    summary = {
        "run_name": str(run_name),
        "run_id": str(run_id or run_name),
        "group": str(getattr(cfg, "group", "")),
        "variant": str(vname),
        "select_by": str(getattr(cfg, "select_by", "mse")),
        "best_epoch": int(best_epoch),
        "best_val_mse": float(best_val),
        "best_by_mse": None if best_by_mse is None else {
            "epoch": int(best_by_mse.get("epoch", -1)),
            "val_mse": float(best_by_mse.get("val_mse", np.nan)),
            "val_loss": float(best_by_mse.get("val_loss", np.nan)),
            "val_mu2_mean": float(best_by_mse.get("val_mu2_mean", np.nan)),
            "val_frac_too_low": float(best_by_mse.get("val_frac_too_low", np.nan)),
            "val_frac_too_high": float(best_by_mse.get("val_frac_too_high", np.nan)),
        },
        "best_by_probml_proxy": None if best_by_probml is None else {
            "epoch": int(best_by_probml.get("epoch", -1)),
            "val_mse": float(best_by_probml.get("val_mse", np.nan)),
            "val_loss": float(best_by_probml.get("val_loss", np.nan)),
            "score_probml_proxy": float(best_by_probml.get("val_loss", np.nan)),
            "val_mu2_mean": float(best_by_probml.get("val_mu2_mean", np.nan)),
            "val_frac_too_low": float(best_by_probml.get("val_frac_too_low", np.nan)),
            "val_frac_too_high": float(best_by_probml.get("val_frac_too_high", np.nan)),
        },
        "final_test": final_test if isinstance(final_test, dict) else {},
    }

    with open(os.path.join(out_dir, "cfg.json"), "w", encoding="utf-8") as f:
        json.dump(asdict(cfg), f, indent=2, ensure_ascii=False)
    df_epochs.to_csv(os.path.join(out_dir, "history.csv"), index=False)
    with open(os.path.join(out_dir, "summary.json"), "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)

    # Optional inline plots (no saving)
    if show_plots:
        import matplotlib.pyplot as plt

        def _plot_series(col):
            if col not in df_epochs.columns:
                return
            plt.figure()
            plt.plot(df_epochs["epoch"].values, df_epochs[col].astype(float).values)
            plt.xlabel("epoch")
            plt.ylabel(col)
            plt.title(f"{run_name} — {col}")
            plt.tight_layout()
            plt.show()

        for col in ["val_mse", "val_loss", "val_mu2_mean", "val_frac_too_low", "val_frac_too_high"]:
            _plot_series(col)

    return summary, df_epochs

def run_experiment(group: str, base_cfg, overrides=None, seed: int = 0, verbose: bool = True, out_dir=None):
    "One run wrapper used by the suite loop."
    overrides = {} if overrides is None else dict(overrides)
    group2 = _group_alias(group)
    cfg2 = cfg_copy(base_cfg, group=group2, **overrides)

    train_loader, val_loader, test_loader = build_loaders(group2, cfg2, seed=seed)
    # paper-suite: disable pruning (projection variants need time)
    res = run_trial(cfg2, seed=seed, train_loader=train_loader, val_loader=val_loader, test_loader=test_loader, out_dir=out_dir, prune=False)

    summary0 = res["summary"]
    history0 = res["history"]

    best_epoch0 = int(summary0["best_epoch"])
    best_val_mse0 = float(summary0["best_val_mse"])
    best_score0 = float(summary0.get("best_score", np.nan))

    final_test0 = {}
    for row in history0:
        if int(row.get("epoch", -1)) == best_epoch0:
            for k, v in row.items():
                if k.startswith("test_") and isinstance(v, (int, float, np.floating)):
                    final_test0[k.replace("test_", "")] = float(v)
            break

    out = {
        "cfg": cfg2,
        "history": history0,
        "best_epoch": best_epoch0,
        "best_val_mse": best_val_mse0,
        "best_score": best_score0,
        "final_test": final_test0,
    }
    if verbose:
        print(f"[run_experiment] group={group2} seed={seed} best_epoch={best_epoch0} best_val_mse={best_val_mse0:.6g} best_score={best_score0:.6g}")
    return out

def _meanstd(df, group_cols, metric_cols):
    g = df.groupby(group_cols, dropna=False)[metric_cols].agg(['mean','std']).reset_index()
    # flatten columns
    new_cols = []
    for c in g.columns:
        if isinstance(c, tuple):
            a,b = c
            if b in ("mean","std"):
                new_cols.append(f"{a}_{b}")
            else:
                new_cols.append(str(a))
        else:
            new_cols.append(str(c))
    g.columns = new_cols
    return g

# -------------------------
# Suite (multi-seed)
# -------------------------
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)

SUITE_GROUPS = ["ECL", "Traffic", "Exchange", "Weather"]
SEEDS = [0, 1, 2, 3, 4]   # <-- multi-seed reports (ce qui manquait)

base_paper = cfg_copy(cfg,
    select_by="probml",
    epochs=max(int(cfg.epochs), 20),
    early_stop_patience=max(int(cfg.early_stop_patience), 6),
)
setattr(base_paper, "save_pt", False)  # <-- n'écrit pas de .pt (JSON/CSV only)

VARIANTS = [
    ("homeo",    {"use_neuron_regulator": True,  "use_projection": False}),
    ("projOFF",  {"use_neuron_regulator": False, "use_projection": False}),
    ("projON",   {"use_neuron_regulator": False, "use_projection": True, "proj_every": 1}),
]

os.makedirs("autopilot_reports", exist_ok=True)
rows_all = []

for seed in SEEDS:
    for g in SUITE_GROUPS:
        for name, ov in VARIANTS:
            run_name = f"{_group_alias(g)}_H{base_paper.horizon}_in{base_paper.input_len}_{name}"
            run_id   = f"{run_name}_seed{seed}"

            print("\n==============================")
            print("RUN:", run_id)
            print("==============================")

            out = run_experiment(g, base_paper, overrides=ov, seed=seed, verbose=True,
                                 out_dir=os.path.join("autopilot_reports", run_id, "raw"))

            summary, df_epochs = autopilot_report(
                out["cfg"], out["history"],
                best_epoch=out["best_epoch"],
                best_val=out["best_val_mse"],
                final_test=out["final_test"],
                run_name=run_name,
                run_id=run_id,
                variant_name=name,
                out_root="autopilot_reports",
                show_plots=False,
            )

            row = {
                "seed": int(seed),
                "group": out["cfg"].group,
                "variant": name,
                "best_epoch": int(out["best_epoch"]),
                "best_val_mse": float(out["best_val_mse"]),
                "best_score": float(out["best_score"]),
                "best_by_mse_val_mse": float(summary["best_by_mse"]["val_mse"]) if summary.get("best_by_mse") else float("nan"),
                "best_by_probml_val_mse": float(summary["best_by_probml_proxy"]["val_mse"]) if summary.get("best_by_probml_proxy") else float("nan"),
                "best_by_probml_score": float(summary["best_by_probml_proxy"]["score_probml_proxy"]) if summary.get("best_by_probml_proxy") else float("nan"),
            }
            if isinstance(out["final_test"], dict):
                for k, v in out["final_test"].items():
                    if isinstance(v, (int, float, np.floating)):
                        row[f"test_{k}"] = float(v)

            # derived internal diagnostic: out = low + high
            if ("test_frac_too_low" in row) and ("test_frac_too_high" in row):
                row["test_out"] = float(row["test_frac_too_low"] + row["test_frac_too_high"])

            rows_all.append(row)

df_all = pd.DataFrame(rows_all).sort_values(["seed","group","variant"]).reset_index(drop=True)

# (A) Affichage complet (ce que Kaggle doit au moins montrer)
print("\n\n=== PAPER SUITE (PER-SEED) ===")
display(df_all)
print(df_all.to_string(index=False))

# (B) Sauvegardes CSV par seed + global
df_all.to_csv("autopilot_reports/paper_suite_aggregate_allseeds.csv", index=False)
for seed in SEEDS:
    df_s = df_all[df_all["seed"] == seed].copy()
    df_s.to_csv(f"autopilot_reports/paper_suite_aggregate_seed{seed}.csv", index=False)

print("Saved:",
      "autopilot_reports/paper_suite_aggregate_allseeds.csv",
      "and per-seed CSVs.")

# (C) Mean±std (ce que la Section 5/Table 1 veut)
metric_cols = [c for c in df_all.columns if c.startswith("test_")] + ["best_val_mse", "best_by_probml_score"]
metric_cols = [c for c in metric_cols if c in df_all.columns]

df_meanstd = _meanstd(df_all, group_cols=["group","variant"], metric_cols=metric_cols).sort_values(["group","variant"]).reset_index(drop=True)
print("\n\n=== PAPER SUITE (MEAN ± STD across seeds) ===")
display(df_meanstd)
print(df_meanstd.to_string(index=False))
df_meanstd.to_csv("autopilot_reports/paper_suite_aggregate_meanstd.csv", index=False)
print("Saved:", "autopilot_reports/paper_suite_aggregate_meanstd.csv")

# (D) Variante sélectionnée (min score probml proxy) par (group, seed)
sel_rows = []
for (seed, group), dfg in df_all.groupby(["seed","group"]):
    if "best_by_probml_score" in dfg.columns and dfg["best_by_probml_score"].notna().any():
        j = int(dfg["best_by_probml_score"].astype(float).idxmin())
    else:
        j = int(dfg["best_val_mse"].astype(float).idxmin())
    r = dfg.loc[j].to_dict()
    sel_rows.append({
        "seed": int(seed),
        "group": group,
        "selected_variant": r["variant"],
        "selected_best_by_probml_score": float(r.get("best_by_probml_score", np.nan)),
        "selected_test_mse": float(r.get("test_mse", np.nan)),
        "selected_test_mae": float(r.get("test_mae", np.nan)),
        "selected_test_out": float(r.get("test_out", np.nan)),
        "selected_test_mu2_mean": float(r.get("test_mu2_mean", np.nan)),
        "selected_test_kl": float(r.get("test_kl", np.nan)),
    })

df_selected = pd.DataFrame(sel_rows).sort_values(["group","seed"]).reset_index(drop=True)
print("\n\n=== SELECTED VARIANT per group/seed (by probml proxy) ===")
display(df_selected)
df_selected.to_csv("autopilot_reports/paper_suite_selected_by_seed.csv", index=False)
print("Saved:", "autopilot_reports/paper_suite_selected_by_seed.csv")


# Correlation sanity-check: do internal metrics predict test MSE?
try:
    corr_metrics = ["test_out", "test_mu2_mean", "test_kl", "test_ar_penalty", "test_ar_loss_share"]
    corr_rows = []
    for m in corr_metrics:
        if m not in df_all.columns: 
            continue
        x = pd.to_numeric(df_all[m], errors="coerce").to_numpy()
        y = pd.to_numeric(df_all["test_mse"], errors="coerce").to_numpy()
        mask = np.isfinite(x) & np.isfinite(y)
        r = float(np.corrcoef(x[mask], y[mask])[0,1]) if mask.sum() >= 3 else np.nan
        corr_rows.append({"metric": m, "pearson_r": r, "n": int(mask.sum())})
    df_corr = pd.DataFrame(corr_rows)
    print("\n\n=== PAPER SUITE: Correlation (internal metric vs test_mse) ===")
    display(df_corr)
    print(df_corr.to_string(index=False))
    df_corr.to_csv(os.path.join(REPORTS_DIR, "paper_suite_corr.csv"), index=False)
    print("Saved:", os.path.join(REPORTS_DIR, "paper_suite_corr.csv"))
except Exception as e:
    print("[warn] correlation block failed:", repr(e))



RUN: ECL_H96_in336_homeo_seed0


/tmp/ipykernel_55/3029606597.py:161: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


[run_experiment] group=ECL seed=0 best_epoch=3 best_val_mse=0.202129 best_score=0.456602

RUN: ECL_H96_in336_projOFF_seed0


/tmp/ipykernel_55/3029606597.py:161: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


[run_experiment] group=ECL seed=0 best_epoch=18 best_val_mse=0.188946 best_score=0.400655

RUN: ECL_H96_in336_projON_seed0


...

   group variant  test_loss_mean  test_loss_std  test_mse_mean  test_mse_std  test_mae_mean  test_mae_std  test_kl_mean  test_kl_std  test_band_mean  test_band_std  test_ar_mean  test_ar_std  test_mu2_mean_mean  test_mu2_mean_std  test_frac_too_low_mean  test_frac_too_low_std  test_frac_too_high_mean  test_frac_too_high_std  test_out_mean  test_out_std  best_val_mse_mean  best_val_mse_std  best_by_probml_score_mean  best_by_probml_score_std
     ECL   homeo        0.473411       0.018678       0.217084      0.006846       0.331584      0.003282     74.518525    20.845144        0.060415       0.012070           0.0          0.0            0.117555           0.011136                0.309050               0.070680                 0.053204                0.003644       0.362253      0.072009           0.206221          0.005018                   0.427784                  0.020583
     ECL projOFF        0.482303       0.013180       0.214333      0.008942       0.332013      0.009391     

,seed,group,selected_variant,selected_best_by_probml_score,selected_test_mse,selected_test_mae,selected_test_out,selected_test_mu2_mean,selected_test_kl
0,0,ECL,projOFF,0.400655,0.199649,0.316067,0.567721,0.133310,72.569982
1,1,ECL,projOFF,0.393390,0.222101,0.338418,0.397722,0.085765,47.919100
2,2,ECL,projOFF,0.395328,0.212336,0.330945,0.518216,0.093895,52.818753
3,3,ECL,projOFF,0.394703,0.218458,0.337397,0.468941,0.086972,48.832805
4,4,ECL,projOFF,0.401192,0.219119,0.337240,0.463669,0.086841,48.711767
5,0,Exchange,projON,16.910547,0.757082,0.674410,0.467494,0.087898,291.459014
6,1,Exchange,projOFF,26.082916,0.149849,0.295249,0.601144,0.147931,236.146007
7,2,Exchange,homeo,16.505481,0.165791,0.313721,0.420340,0.153867,253.187040
8,3,Exchange,homeo,17.136862,0.119405,0.263254,0.466378,0.172351,255.192932
9,4,Exchange,projOFF,15.917975,0.265521,0.392984,0.531390,0.110400,219.132296


Saved: autopilot_reports/paper_suite_selected_by_seed.csv


=== PAPER SUITE: Correlation (internal metric vs test_mse) ===


,metric,pearson_r,n
0,test_out,-0.256247,60
1,test_mu2_mean,0.056921,60
2,test_kl,0.292283,60


       metric  pearson_r  n
     test_out  -0.256247 60
test_mu2_mean   0.056921 60
      test_kl   0.292283 60
[warn] correlation block failed: NameError("name 'REPORTS_DIR' is not defined")


In [ ]:
# ============================================================
# (16d) Ablation: deterministic vs nVAE (VDN) — now for multiple groups (ECL / TrafficL / Weather)
#  - JSON/CSV only
#  - IMPORTANT Kaggle: on *affiche* les tableaux (si zip manquant)
# ============================================================
import os, json
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

def _extract_test_metrics_at_epoch(history, epoch: int):
    out = {}
    for row in history:
        if int(row.get("epoch", -1)) == int(epoch):
            for k, v in row.items():
                if k.startswith("test_") and isinstance(v, (int, float, np.floating)):
                    out[k.replace("test_", "")] = float(v)
            break
    return out

# Config base (reprend base_paper si la cellule suite a été exécutée)
if "base_paper" in globals():
    base_ablation = base_paper
    setattr(base_ablation, "save_pt", False)
else:
    base_ablation = cfg_copy(cfg, select_by="probml",
                             epochs=max(int(cfg.epochs), 20),
                             early_stop_patience=max(int(cfg.early_stop_patience), 6))

ABLATION_GROUPS = ["ECL", "TrafficL", "Weather"]  # <-- ce qui manquait (run_det/run_nvae)
ABLATION_SEEDS  = [0, 1, 2, 3, 4]  # 5 seeds (modifie si besoin)

os.makedirs("ablation_runs", exist_ok=True)
rows = []

for seed in ABLATION_SEEDS:
    for g in ABLATION_GROUPS:
        group = _group_alias(g)

        train_loader, val_loader, test_loader = build_loaders(group, base_ablation, seed=seed)

        # nVAE / VDN
        cfg_nvae = cfg_copy(base_ablation, group=group, model="nvae")
        setattr(cfg_nvae, "save_pt", False)
        out_dir_nvae = os.path.join("ablation_runs", f"{group}_seed{seed}", "run_nvae")
        res_nvae = run_trial(cfg_nvae, seed=seed, train_loader=train_loader, val_loader=val_loader, test_loader=test_loader,
                             out_dir=out_dir_nvae, prune=False)

        sum_nvae = res_nvae["summary"]
        hist_nvae = res_nvae["history"]
        test_nvae = _extract_test_metrics_at_epoch(hist_nvae, sum_nvae["best_epoch"])

        # deterministic baseline
        cfg_det = cfg_copy(base_ablation, group=group, model="det")
        setattr(cfg_det, "save_pt", False)
        out_dir_det = os.path.join("ablation_runs", f"{group}_seed{seed}", "run_det")
        res_det = run_trial(cfg_det, seed=seed, train_loader=train_loader, val_loader=val_loader, test_loader=test_loader,
                            out_dir=out_dir_det, prune=False)

        sum_det = res_det["summary"]
        hist_det = res_det["history"]
        test_det = _extract_test_metrics_at_epoch(hist_det, sum_det["best_epoch"])

        # Row builder (test metrics are what paper wants)
        def _row(model_name, summary, test_dict):
            r = {
                "seed": int(seed),
                "group": group,
                "model": model_name,
                "best_epoch": int(summary["best_epoch"]),
                "best_val_mse": float(summary["best_val_mse"]),
                "select_by": str(summary.get("select_by", getattr(base_ablation, "select_by", "probml"))),
            }
            # Extras from summary (for VDN only; det will be NaN later)
            for k in ["beta_mean","beta_min","beta_max","kl_mean_unit","lam_ar_final","secs"]:
                if k in summary and isinstance(summary[k], (int, float, np.floating)):
                    r[k] = float(summary[k])

            # test metrics (extract from history at best epoch)
            for k, v in (test_dict or {}).items():
                r[f"test_{k}"] = float(v)
            if ("test_frac_too_low" in r) and ("test_frac_too_high" in r):
                r["test_out"] = float(r["test_frac_too_low"] + r["test_frac_too_high"])
            return r

        rows.append(_row("VDN (nVAE-1D)", sum_nvae, test_nvae))
        rows.append(_row("Deterministic", sum_det, test_det))

df_ab = pd.DataFrame(rows).sort_values(["group","seed","model"]).reset_index(drop=True)

# Reviewer-proof: internal metrics for deterministic are N/A (not '0')
internal_cols = ["test_kl", "test_mu2_mean", "test_frac_too_low", "test_frac_too_high", "test_out", "test_ar"]
mask_det = df_ab["model"].str.contains("Deterministic", na=False)
for c in internal_cols:
    if c in df_ab.columns:
        df_ab.loc[mask_det, c] = np.nan

print("\n\n=== ABLATION det vs VDN (PER-SEED) ===")
display(df_ab)
print(df_ab.to_string(index=False))

df_ab.to_csv("ablation_runs/ablation_det_vs_vdn_all.csv", index=False)
print("Saved:", "ablation_runs/ablation_det_vs_vdn_all.csv")

# Optional mean±std (if ABLATION_SEEDS has >1)
if len(ABLATION_SEEDS) > 1:
    metric_cols = [c for c in df_ab.columns if c.startswith("test_")] + ["best_val_mse"]
    metric_cols = [c for c in metric_cols if c in df_ab.columns]
    df_ms = _meanstd(df_ab, group_cols=["group","model"], metric_cols=metric_cols).sort_values(["group","model"]).reset_index(drop=True)
    print("\n\n=== ABLATION det vs VDN (MEAN ± STD across seeds) ===")
    display(df_ms)
    print(df_ms.to_string(index=False))
    df_ms.to_csv("ablation_runs/ablation_det_vs_vdn_meanstd.csv", index=False)
    print("Saved:", "ablation_runs/ablation_det_vs_vdn_meanstd.csv")


/tmp/ipykernel_55/3029606597.py:161: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
/tmp/ipykernel_55/3029606597.py:161: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
/tmp/ipykernel_55/3029606597.py:161: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
/tmp/ipykernel_55/3029606597.py:161: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
/tmp/ipykernel_55/3029606597.py:161: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
/tmp/ipykernel_55/3029606597.p

In [ ]:

from pathlib import Path
import zipfile
import time

ROOT = Path("/kaggle/working/ablation_runs")
EXTS = {".json", ".csv"}

# Nom du zip (dans /kaggle/working pour être téléchargeable)
ts = time.strftime("%Y%m%d_%H%M%S")
ZIP_PATH = ROOT / f"ablation_runs{ts}.zip"

def should_include(p: Path) -> bool:
    return p.is_file() and (p.suffix.lower() in EXTS)

# Collecte
files = [p for p in ROOT.rglob("*") if should_include(p)]

# Évite de zipper un zip existant si jamais l'extension est dans EXTS (ce n'est pas le cas ici)
files = [p for p in files if p.resolve() != ZIP_PATH.resolve()]

print(f"ROOT      : {ROOT}")
print(f"ZIP_PATH  : {ZIP_PATH}")
print(f"Found     : {len(files)} files")
print("Examples  :", [str(p.relative_to(ROOT)) for p in files[:10]])

# Zip en conservant l'arborescence
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for p in files:
        arcname = p.relative_to(ROOT)  # <-- conserve la structure des dossiers
        zf.write(p, arcname=str(arcname))

print("Done. ZIP created:", ZIP_PATH)